# 05 — Layer 3 Portfolio Construction

This notebook compares **portfolio construction methods**, not new alphas. The goal is to answer a practical question:

> Given the Layer 2 candidate sleeves and the Layer 2B risk / regime engine, which portfolio construction methods look most robust **out of sample** once we account for estimation error, turnover, transaction costs, concentration, and drawdown control?

The notebook therefore focuses on:

- a **small shortlist of credible Layer 2 sleeves**
- realistic **long-only constrained allocators**
- transparent **risk overlays**
- walk-forward, out-of-sample comparison
- honest discussion of where classical optimizers become fragile


## 1. Global Design, Timing, and Why Layer 3 Is Sleeve-Based

Layer 3 sits on top of the project's existing research stack:

- **Layer 1** produces tradable alpha signals
- **Layer 2A** translates those signals into investable ETF sleeves
- **Layer 2B** produces regime states, covariance inputs, volatility, drawdown, and defensive overlays

The most important design choice here is that we **allocate across a shortlist of Layer 2 sleeves first**, then look through those sleeve allocations into final ETF weights.

That is deliberate.

A direct optimizer across every ETF, every signal, and every regime feature would create a larger search space, more unstable inputs, and more ways to overfit. A sleeve-based construction layer is usually more realistic for a solo quant workflow because:

- the sleeves already encode economic logic
- the sleeve universe is small enough to diagnose
- the resulting optimizer is easier to regularize
- the final output can still be converted into **actual ETF weights**

**Timing convention**

- The notebook preserves the project's **weekly data framing**
- Portfolio construction is evaluated at **weekly return frequency**
- The default **rebalance frequency is monthly** unless stated otherwise
- All conviction inputs are based on **tradable Layer 1 / Layer 2 outputs only**
- All regime features are consumed in their **already-lagged tradable form**
- No same-date forward returns are used when forming weights

In plain English: if the system could not have known it by the rebalance date, Layer 3 does not get to use it.


In [ ]:
from __future__ import annotations

import json
import math
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from scipy.cluster.hierarchy import fcluster, leaves_list, linkage
    from scipy.optimize import minimize
    from scipy.spatial.distance import squareform
    from scipy.stats import norm
except Exception:
    fcluster = None
    leaves_list = None
    linkage = None
    minimize = None
    squareform = None
    norm = None

try:
    import cvxpy as cp
except Exception:
    cp = None

try:
    from sklearn.covariance import LedoitWolf
except Exception:
    LedoitWolf = None

warnings.filterwarnings("ignore")
pd.options.display.float_format = "{:,.4f}".format
plt.rcParams["figure.figsize"] = (13, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
np.random.seed(7)

ROOT = Path(".").resolve()
DATA_ROOT = ROOT / "data"
OUTPUT_DIR = DATA_ROOT / "05_layer3_portfolio_construction"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_HUB_CANDIDATES = [
    DATA_ROOT / "01_data_hub",
    DATA_ROOT / "processed",
    ROOT / "data" / "processed",
    ROOT / "data2" / "processed",
]
LAYER1_CANDIDATES = [
    DATA_ROOT / "02_layer1_signals",
    DATA_ROOT / "processed",
    ROOT / "data" / "processed",
]
LAYER2A_CANDIDATES = [
    DATA_ROOT / "03_layer2a_strategy_logic",
]
LAYER2B_CANDIDATES = [
    DATA_ROOT / "04_layer2b_risk_regime_engine",
]

REQUIRED_DATA_HUB = [
    "weekly_prices.csv",
    "weekly_returns.csv",
    "daily_returns.csv",
    "universe.json",
    "proxy_mapping.json",
    "market_proxy_weekly.csv",
]
REQUIRED_LAYER1 = [
    "signal_xsmom.csv",
    "signal_multi_horizon_mom.csv",
    "signal_residual_momentum.csv",
    "signal_quality.csv",
    "signal_value.csv",
    "signal_bab.csv",
    "signal_carry.csv",
    "signal_manifest.json",
]
REQUIRED_LAYER2A = [
    "strategy_summary_table.csv",
    "layer2_manifest.json",
]
REQUIRED_LAYER2B = [
    "regime_states.csv",
    "regime_score.csv",
    "covariance_snapshots_long.csv",
]

SIGNAL_LAG_WEEKS = 1
EXTERNAL_DATA_LAG_WEEKS = 1
WEEKS_PER_YEAR = 52
TRADING_DAYS_PER_YEAR = 252

REBALANCE_FREQUENCY = "monthly"
TRAIN_WINDOW_WEEKS = 156
ALT_TRAIN_WINDOW_WEEKS = 104
MIN_TRAIN_OBS = 78
DEFAULT_COST_BPS = 10
COST_BPS_GRID = [0, 5, 10, 25]

TARGET_VOL_ANN = 0.12
TARGET_VOL_FLOOR = 0.35
# Layer 3 is the final portfolio-aware allocator, so the target-vol cap stays at 1.00; Layer 2B's looser cap is only a market-proxy regime scalar.
TARGET_VOL_CEIL = 1.00
MAX_SLEEVE_WEIGHT = 0.45
MAX_ETF_WEIGHT = 0.35
SLEEVE_REALLOCATION_SPEED = 0.40
MEAN_RETURN_SCALE_ANN = 0.08
RISK_AVERSION = 6.0
TURNOVER_PENALTY = 0.01
BL_TAU = 0.05
BL_RISK_AVERSION = 2.5
BL_CONFIDENCE_SPREAD_DIVISOR = 2.0
BL_CONFIDENCE_FLOOR = 0.25
BL_CONFIDENCE_CEIL = 0.90
CVAR_LEVEL = 0.95
SUBPERIODS = 3
EULER_GAMMA = 0.5772156649

CANDIDATE_SLEEVES = [
    ("dual_momentum_topn", "core"),
    ("cta_trend_long_only", "core"),
    ("composite_equal_weight", "core"),
    ("composite_regime_conditioned", "core"),
    ("taa_10m_sma", "core"),
    ("composite_breadth_filtered", "experimental"),
]

SIGNAL_COLUMN_MAP = {
    "xsmom_global": ("signal_xsmom.csv", "xsmom_score_tradable"),
    "multi_mom_invvol": ("signal_multi_horizon_mom.csv", "multi_mom_invvol_score_tradable"),
    "residual_momentum": ("signal_residual_momentum.csv", "residual_mom_score_tradable"),
    "quality_proxy": ("signal_quality.csv", "quality_score_tradable"),
    "value_proxy": ("signal_value.csv", "value_score_tradable"),
    "carry_proxy": ("signal_carry.csv", "carry_score_tradable"),
    "bab_proxy": ("signal_bab.csv", "bab_score_tradable"),
    "reversal_4w_global": ("signal_reversal.csv", "reversal_4w_score_tradable"),
}

EXPECTED_RETURN_HISTORY_NAMES = {
    "ic_signal": "composite_ic_weighted",
    "regime_signal": "composite_regime_conditioned",
}

print("Layer 3 output directory:", OUTPUT_DIR.resolve())
print("Monthly rebalancing on weekly data with train window:", TRAIN_WINDOW_WEEKS, "weeks")


## 2. Research Anchors and Why These Methods Were Chosen

The notebook uses a deliberately mixed comparison set:

- **Simple baselines** that are hard to beat out of sample
- **Classical optimizers** that are economically intuitive but error-sensitive
- **Risk-based methods** that often behave better when expected returns are noisy
- **A few carefully selected extensions** that are genuinely useful for ETF research

The implementation choices below are informed by the following literature and practitioner references:

- [Markowitz (1952), *Portfolio Selection*](https://www.jstor.org/stable/2975974) for mean-variance optimization
- [Ledoit and Wolf (2004), *Honey, I Shrunk the Sample Covariance Matrix*](https://www.ledoit.net/honey.pdf) for shrinkage-based covariance stabilization
- [Idzorek (2004/2007), *A Step-By-Step Guide to the Black-Litterman Model*](https://docslib.org/doc/8265707/a-step-by-step-guide-to-the-black-litterman-model) for a practical Black-Litterman implementation
- [López de Prado (2016), *Building Diversified Portfolios that Outperform Out of Sample*](https://doi.org/10.3905/jpm.2016.42.4.059) for HRP
- [Raffinot (2018), *The Hierarchical Equal Risk Contribution Portfolio*](https://ssrn.com/abstract=3237540) for HERC
- [Choueifaty and Coignard (2008), *Toward Maximum Diversification*](https://www.tobam.fr/wp-content/uploads/2014/12/TOBAM-JoPM-Maximum-Div-2008.pdf) for diversification-ratio allocation
- [Rockafellar and Uryasev (2000), *Optimization of Conditional Value-at-Risk*](https://pdf4pro.com/file/2b608/public_files_docs_public_finance_Active_Risk_Management_Uryasev_Rockafellar__Optimization_CVaR.pdf.pdf) for tail-risk-aware optimization
- [Moreira and Muir (2017), *Volatility Managed Portfolios*](https://www.nber.org/papers/w22208) for volatility-aware exposure scaling
- [Bailey and López de Prado, probabilistic / deflated Sharpe diagnostics](https://ssrn.com/abstract=2460551) for multiple-testing-aware performance interpretation

Two methods were **considered but not implemented as baselines**:

- **Nested Cluster Optimization (NCO)**: useful when the asset or sleeve universe is much larger; with a deliberately small sleeve set, the added complexity is unlikely to pay for itself yet.
- **Fully Bayesian / resampled optimizers**: valuable in principle, but they add another large layer of modeling assumptions before the current stack has earned that complexity.


In [ ]:
# Small technical-debt note: these basic IO / annualization helpers still overlap with Layer 2 notebooks. Leaving them local keeps the notebook easy to run top-to-bottom, but a tiny shared module remains a future cleanup item.
def flatten_columns(df):
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ["_".join([str(x) for x in col if x not in [None, ""]]) for col in df.columns]
    df.columns = [str(c) for c in df.columns]
    df.columns.name = None
    df.index = pd.to_datetime(df.index).tz_localize(None)
    df.index.name = "Date"
    return df


def choose_input_dir(candidates, required_files):
    best_dir = None
    best_count = -1
    best_missing = list(required_files)
    for directory in candidates:
        existing = [f for f in required_files if (directory / f).exists()]
        if len(existing) > best_count:
            best_dir = directory
            best_count = len(existing)
            best_missing = [f for f in required_files if f not in existing]
    return best_dir, best_missing


def read_panel_csv(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    frame = pd.read_csv(path, index_col=0, parse_dates=True)
    frame.index = pd.to_datetime(frame.index).tz_localize(None)
    frame.index.name = "Date"
    return flatten_columns(frame)


def read_table_csv(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)


def read_json_file(path):
    path = Path(path)
    if not path.exists():
        return {}
    return json.loads(path.read_text())


def read_signal_long(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_csv(path, parse_dates=["Date"])
    df["Date"] = pd.to_datetime(df["Date"]).dt.tz_localize(None)
    return df


def read_strategy_positions(path):
    return read_panel_csv(path)


def read_strategy_returns(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    frame = pd.read_csv(path, index_col=0, parse_dates=True)
    frame.index = pd.to_datetime(frame.index).tz_localize(None)
    frame.index.name = "Date"
    return flatten_columns(frame)


def long_signal_to_panel(df, value_col, index=None, columns=None):
    if df.empty or value_col not in df.columns:
        if index is None or columns is None:
            return pd.DataFrame()
        return pd.DataFrame(np.nan, index=index, columns=columns)
    panel = df.pivot(index="Date", columns="Ticker", values=value_col).sort_index()
    panel.index = pd.to_datetime(panel.index).tz_localize(None)
    if index is not None:
        panel = panel.reindex(index)
    if columns is not None:
        panel = panel.reindex(columns=columns)
    panel.index.name = "Date"
    panel.columns.name = None
    return panel


def get_proxy_ticker(proxy_mapping, role, fallback=None):
    if not isinstance(proxy_mapping, dict):
        return fallback
    entry = proxy_mapping.get(role, fallback)
    if isinstance(entry, dict):
        return entry.get("ticker", fallback)
    if isinstance(entry, str):
        return entry
    return fallback


def last_friday_of_month_mask(index):
    month_key = pd.Series(pd.Index(index).to_period("M").astype(str), index=index)
    mask = month_key.ne(month_key.shift(-1))
    if len(mask) > 0:
        mask.iloc[0] = True
    return mask


def rebalance_mask(index, frequency):
    frequency = str(frequency).lower()
    if frequency.startswith("w"):
        return pd.Series(True, index=index)
    if frequency.startswith("m"):
        return last_friday_of_month_mask(index)
    raise ValueError(f"Unsupported rebalance frequency: {frequency}")


def safe_inverse(values, floor=1e-12):
    values = pd.Series(values, dtype=float)
    return 1.0 / values.replace(0.0, np.nan).clip(lower=floor)


def row_zscore(frame, clip_value=3.0):
    out = []
    for _, row in frame.iterrows():
        valid = row.dropna()
        scored = pd.Series(np.nan, index=row.index, dtype=float)
        if len(valid) == 1:
            scored.loc[valid.index] = 0.0
        elif len(valid) > 1:
            z = (valid - valid.mean()) / (valid.std(ddof=1) + 1e-12)
            scored.loc[valid.index] = z.clip(-clip_value, clip_value)
        out.append(scored)
    return pd.DataFrame(out, index=frame.index)


def normalize_long_only(weights, max_weight=MAX_SLEEVE_WEIGHT):
    weights = pd.Series(weights, dtype=float).clip(lower=0.0).fillna(0.0)
    if weights.sum() <= 0:
        weights[:] = 1.0 / len(weights) if len(weights) else 0.0
        return weights
    weights = weights / weights.sum()
    for _ in range(25):
        over = weights > max_weight
        if not over.any():
            break
        excess = (weights[over] - max_weight).sum()
        weights.loc[over] = max_weight
        under = weights < max_weight - 1e-12
        if under.any() and excess > 0:
            weights.loc[under] += excess * weights.loc[under] / weights.loc[under].sum()
        elif excess > 0:
            weights += excess / len(weights)
        weights = weights.clip(lower=0.0)
        weights = weights / weights.sum()
    return weights / weights.sum()


def annualized_return(return_series, periods_per_year=WEEKS_PER_YEAR):
    returns = pd.Series(return_series).dropna()
    if returns.empty:
        return np.nan
    wealth = (1.0 + returns).cumprod()
    return wealth.iloc[-1] ** (periods_per_year / len(returns)) - 1.0


def annualized_vol(return_series, periods_per_year=WEEKS_PER_YEAR):
    returns = pd.Series(return_series).dropna()
    if len(returns) < 2:
        return np.nan
    return returns.std(ddof=1) * np.sqrt(periods_per_year)


def conditional_var(return_series, level=0.05):
    returns = pd.Series(return_series).dropna()
    if returns.empty:
        return np.nan
    cutoff = returns.quantile(level)
    tail = returns[returns <= cutoff]
    return tail.mean() if len(tail) else np.nan


def effective_n(weight_panel):
    if weight_panel is None or weight_panel.empty:
        return np.nan
    denom = weight_panel.pow(2).sum(axis=1).replace(0.0, np.nan)
    return (1.0 / denom).mean()


def avg_hhi(weight_panel):
    if weight_panel is None or weight_panel.empty:
        return np.nan
    return weight_panel.pow(2).sum(axis=1).mean()


def weight_instability(weight_panel):
    if weight_panel is None or weight_panel.empty:
        return np.nan
    return weight_panel.diff().abs().sum(axis=1).mean()


def probabilistic_sharpe_ratio(return_series, benchmark_sr=0.0):
    returns = pd.Series(return_series).dropna()
    if len(returns) < 3:
        return np.nan
    std = returns.std(ddof=1)
    if std == 0 or pd.isna(std):
        return np.nan
    sr = returns.mean() / std
    skew_val = returns.skew()
    kurt_val = returns.kurtosis() + 3.0
    denom_term = 1.0 - skew_val * sr + ((kurt_val - 1.0) / 4.0) * sr ** 2
    if denom_term <= 0:
        return np.nan
    z = (sr - benchmark_sr) * np.sqrt(len(returns) - 1) / np.sqrt(denom_term)
    if norm is not None:
        return float(norm.cdf(z))
    return float(0.5 * (1.0 + math.erf(z / np.sqrt(2.0))))


def selection_aware_sharpe_hurdle(return_series, trials):
    returns = pd.Series(return_series).dropna()
    if len(returns) < 3 or trials < 2 or norm is None:
        return np.nan
    std = returns.std(ddof=1)
    if std == 0 or pd.isna(std):
        return np.nan
    sr = returns.mean() / std
    skew_val = returns.skew()
    kurt_val = returns.kurtosis() + 3.0
    sr_std = np.sqrt(max(1e-12, (1.0 - skew_val * sr + ((kurt_val - 1.0) / 4.0) * sr ** 2) / (len(returns) - 1)))
    q1 = norm.ppf(1.0 - 1.0 / trials)
    q2 = norm.ppf(1.0 - 1.0 / (trials * np.e))
    return float(sr_std * ((1.0 - EULER_GAMMA) * q1 + EULER_GAMMA * q2))


def compute_portfolio_path(weights, next_week_returns, transaction_cost_bps=DEFAULT_COST_BPS):
    weights = weights.reindex(index=next_week_returns.index, columns=next_week_returns.columns).fillna(0.0)
    gross_return = (weights * next_week_returns).sum(axis=1)
    turnover = 0.5 * weights.diff().abs().sum(axis=1)
    if len(turnover) > 0:
        turnover.iloc[0] = np.nan
    cost = turnover.fillna(0.0) * (transaction_cost_bps / 10000.0)
    net_return = gross_return - cost
    wealth = (1.0 + net_return.fillna(0.0)).cumprod()
    running_peak = wealth.cummax()
    drawdown = wealth.div(running_peak) - 1.0
    return pd.DataFrame(
        {
            "gross_return": gross_return,
            "net_return": net_return,
            "turnover": turnover,
            "cost": cost,
            "wealth": wealth,
            "drawdown": drawdown,
        }
    )


def summary_metrics(return_series, turnover_series=None, weight_panel=None, allocation_panel=None, trials=1):
    returns = pd.Series(return_series).dropna()
    if returns.empty:
        return {
            "ann_return": np.nan,
            "ann_vol": np.nan,
            "sharpe": np.nan,
            "max_drawdown": np.nan,
            "calmar": np.nan,
            "cvar_5": np.nan,
            "hit_rate": np.nan,
            "avg_weekly_turnover": np.nan,
            "annual_turnover": np.nan,
            "avg_max_weight": np.nan,
            "avg_effective_n": np.nan,
            "avg_hhi": np.nan,
            "weight_instability": np.nan,
            "allocation_instability": np.nan,
            "psr_zero": np.nan,
            "psr_selection": np.nan,
            "observations": 0,
        }
    wealth = (1.0 + returns).cumprod()
    drawdown = wealth.div(wealth.cummax()) - 1.0
    ann_ret = annualized_return(returns)
    ann_vol = annualized_vol(returns)
    sharpe = np.nan if ann_vol in [0, np.nan] or pd.isna(ann_vol) else ann_ret / ann_vol
    max_dd = drawdown.min()
    calmar = np.nan if pd.isna(max_dd) or max_dd == 0 else ann_ret / abs(max_dd)
    psr_hurdle = selection_aware_sharpe_hurdle(returns, trials=max(trials, 2))
    return {
        "ann_return": ann_ret,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_drawdown": max_dd,
        "calmar": calmar,
        "cvar_5": conditional_var(returns, level=0.05),
        "hit_rate": (returns > 0).mean(),
        "avg_weekly_turnover": np.nan if turnover_series is None else pd.Series(turnover_series).mean(),
        "annual_turnover": np.nan if turnover_series is None else pd.Series(turnover_series).mean() * WEEKS_PER_YEAR,
        "avg_max_weight": np.nan if weight_panel is None or weight_panel.empty else weight_panel.max(axis=1).mean(),
        "avg_effective_n": effective_n(weight_panel),
        "avg_hhi": avg_hhi(weight_panel),
        "weight_instability": weight_instability(weight_panel),
        "allocation_instability": weight_instability(allocation_panel),
        "psr_zero": probabilistic_sharpe_ratio(returns, benchmark_sr=0.0),
        "psr_selection": probabilistic_sharpe_ratio(returns, benchmark_sr=psr_hurdle) if pd.notna(psr_hurdle) else np.nan,
        "observations": int(len(returns)),
    }


def cost_sensitivity_table(weights, next_week_returns, method_name, cost_grid=COST_BPS_GRID):
    rows = []
    for cost_bps in cost_grid:
        path = compute_portfolio_path(weights, next_week_returns, transaction_cost_bps=cost_bps)
        metrics = summary_metrics(path["net_return"], turnover_series=path["turnover"], weight_panel=weights)
        metrics.update({"method_name": method_name, "cost_bps": cost_bps})
        rows.append(metrics)
    return pd.DataFrame(rows)


def split_subperiods(index, n_splits=SUBPERIODS):
    if len(index) < n_splits:
        return [index]
    boundaries = np.array_split(np.arange(len(index)), n_splits)
    return [index[chunk] for chunk in boundaries if len(chunk) > 0]


def subperiod_summary(method_name, return_series, n_splits=SUBPERIODS):
    returns = pd.Series(return_series).dropna()
    if returns.empty:
        return pd.DataFrame()
    rows = []
    for i, idx in enumerate(split_subperiods(returns.index, n_splits=n_splits), start=1):
        sample = returns.reindex(idx).dropna()
        if sample.empty:
            continue
        rows.append(
            {
                "method_name": method_name,
                "subperiod": f"split_{i}",
                "ann_return": annualized_return(sample),
                "ann_vol": annualized_vol(sample),
                "sharpe": np.nan if annualized_vol(sample) in [0, np.nan] or pd.isna(annualized_vol(sample)) else annualized_return(sample) / annualized_vol(sample),
                "max_drawdown": ((1.0 + sample).cumprod().div((1.0 + sample).cumprod().cummax()) - 1.0).min(),
            }
        )
    return pd.DataFrame(rows)


def regime_split_summary(method_name, return_series, risk_state):
    returns = pd.Series(return_series).dropna()
    if returns.empty:
        return pd.DataFrame()
    joined = pd.DataFrame({"return": returns}).join(pd.Series(risk_state, name="risk_state"), how="left")
    rows = []
    for state, group in joined.groupby("risk_state"):
        sample = group["return"].dropna()
        if sample.empty:
            continue
        ann_vol = annualized_vol(sample)
        ann_ret = annualized_return(sample)
        rows.append(
            {
                "method_name": method_name,
                "risk_state": state,
                "ann_return": ann_ret,
                "ann_vol": ann_vol,
                "sharpe": np.nan if ann_vol == 0 or pd.isna(ann_vol) else ann_ret / ann_vol,
                "hit_rate": (sample > 0).mean(),
                "observations": len(sample),
            }
        )
    return pd.DataFrame(rows)


In [ ]:
def trailing_compound_return(simple_returns, lookback):
    return (1.0 + simple_returns).rolling(lookback, min_periods=max(4, lookback // 2)).apply(np.prod, raw=True) - 1.0


def mean_pairwise_correlation(frame, window=26):
    # Match Layer 2B's vectorized rolling-correlation path so the diagnostic stays lightweight and definitionally consistent across notebooks.
    def _mean_off_diag(block):
        arr = block.to_numpy(dtype=float, copy=False)
        if arr.ndim != 2 or arr.shape[0] < 2:
            return np.nan
        mask = ~np.eye(arr.shape[0], dtype=bool)
        vals = arr[mask]
        vals = vals[np.isfinite(vals)]
        return float(vals.mean()) if vals.size else np.nan

    frame = pd.DataFrame(frame).replace([np.inf, -np.inf], np.nan)
    out = pd.Series(np.nan, index=frame.index, name="mean_pairwise_corr", dtype=float)
    if frame.empty or frame.shape[1] < 2:
        return out
    min_periods = min(window, max(8, window // 2))
    frame = frame.dropna(axis=1, how="all")
    if frame.shape[1] < 2:
        return out
    rolling_corr = frame.rolling(window, min_periods=min_periods).corr()
    if rolling_corr.empty:
        return out
    avg = rolling_corr.groupby(level=0, sort=False).apply(_mean_off_diag)
    avg = avg.reindex(out.index)
    avg.name = "mean_pairwise_corr"
    return avg


def weight_history_from_long(weight_history_long, strategy_name, dates, signal_names):
    if weight_history_long.empty:
        return pd.DataFrame(np.nan, index=dates, columns=signal_names)
    subset = weight_history_long[weight_history_long["strategy_name"] == strategy_name].copy()
    if subset.empty:
        return pd.DataFrame(np.nan, index=dates, columns=signal_names)
    subset["Date"] = pd.to_datetime(subset["Date"]).dt.tz_localize(None)
    pivot = subset.pivot(index="Date", columns="signal_name", values="weight").sort_index()
    pivot = pivot.reindex(dates).ffill()
    return pivot.reindex(columns=signal_names)


def combine_signal_panels(signal_panels, weight_history=None):
    if not signal_panels:
        return pd.DataFrame()
    index = next(iter(signal_panels.values())).index
    columns = next(iter(signal_panels.values())).columns
    out_rows = []
    for date in index:
        numerator = pd.Series(0.0, index=columns, dtype=float)
        denominator = pd.Series(0.0, index=columns, dtype=float)
        for signal_name, panel in signal_panels.items():
            row = panel.loc[date].reindex(columns)
            signal_weight = 1.0 if weight_history is None else float(weight_history.loc[date, signal_name]) if signal_name in weight_history.columns and pd.notna(weight_history.loc[date, signal_name]) else 0.0
            numerator = numerator.add(row.fillna(0.0) * signal_weight, fill_value=0.0)
            denominator = denominator.add(row.notna().astype(float) * abs(signal_weight), fill_value=0.0)
        out_rows.append(numerator.div(denominator.replace(0.0, np.nan)))
    return pd.DataFrame(out_rows, index=index)


def sleeve_scores_from_asset_scores(asset_score_panel, sleeve_positions):
    rows = {}
    for sleeve_name, position_panel in sleeve_positions.items():
        aligned_pos = position_panel.reindex(index=asset_score_panel.index, columns=asset_score_panel.columns).fillna(0.0)
        valid_mask = asset_score_panel.notna()
        numerator = (aligned_pos * asset_score_panel.fillna(0.0)).sum(axis=1)
        denominator = aligned_pos.abs().where(valid_mask, 0.0).sum(axis=1).replace(0.0, np.nan)
        rows[sleeve_name] = numerator.div(denominator)
    return pd.DataFrame(rows, index=asset_score_panel.index)


def score_row_to_weekly_mu(score_row, train_returns, annual_scale=MEAN_RETURN_SCALE_ANN):
    row = pd.Series(score_row, dtype=float).dropna()
    if row.empty:
        return pd.Series(dtype=float)
    if row.std(ddof=1) == 0 or len(row) == 1:
        z = pd.Series(0.0, index=row.index)
    else:
        z = ((row - row.mean()) / (row.std(ddof=1) + 1e-12)).clip(-2.0, 2.0)
    vol_anchor = train_returns.std(ddof=1).median() if not train_returns.empty else np.nan
    weekly_scale = annual_scale / WEEKS_PER_YEAR
    if pd.notna(vol_anchor):
        weekly_scale = min(weekly_scale, max(0.25 * vol_anchor, weekly_scale / 2.0))
    return z * weekly_scale


def sanitize_train_window(train_returns, min_obs=10, std_floor=1e-10):
    frame = pd.DataFrame(train_returns).copy()
    if frame.empty:
        return frame
    frame = frame.replace([np.inf, -np.inf], np.nan)
    frame = frame.dropna(axis=1, how="all")
    if frame.empty:
        return frame
    counts = frame.count()
    frame = frame.loc[:, counts[counts >= max(2, min_obs)].index]
    if frame.empty:
        return frame
    frame = frame.dropna(how="any")
    if frame.empty:
        return frame
    std = frame.std(ddof=1).replace([np.inf, -np.inf], np.nan)
    frame = frame.loc[:, std[std > std_floor].index]
    return frame


def sanitize_covariance_matrix(cov, var_floor=1e-12):
    cov = pd.DataFrame(cov).copy()
    if cov.empty:
        return cov
    cov = cov.replace([np.inf, -np.inf], np.nan)
    common = cov.index.intersection(cov.columns)
    cov = cov.loc[common, common]
    if cov.empty:
        return cov
    cov = (cov + cov.T) / 2.0
    diag = pd.Series(np.diag(cov.values), index=cov.index).replace([np.inf, -np.inf], np.nan)
    keep = diag[diag > var_floor].index
    cov = cov.loc[keep, keep]
    if cov.empty:
        return cov
    finite_mask = pd.Series(np.isfinite(cov.to_numpy()).all(axis=1), index=cov.index)
    cov = cov.loc[finite_mask[finite_mask].index, finite_mask[finite_mask].index]
    if cov.empty:
        return cov
    diag_idx = np.diag_indices_from(cov.values)
    cov.values[diag_idx] = np.maximum(np.diag(cov.values), var_floor)
    return cov


def estimate_covariance(train_returns, method="ledoit_wolf"):
    train_returns = sanitize_train_window(train_returns, min_obs=min(10, max(len(train_returns), 1)))
    if train_returns.empty:
        return pd.DataFrame()
    if train_returns.shape[1] < 2 or train_returns.shape[0] < 10:
        return sanitize_covariance_matrix(train_returns.cov())
    if method == "ledoit_wolf" and LedoitWolf is not None:
        try:
            lw = LedoitWolf().fit(train_returns.values)
            return sanitize_covariance_matrix(pd.DataFrame(lw.covariance_, index=train_returns.columns, columns=train_returns.columns))
        except Exception:
            pass
    return sanitize_covariance_matrix(train_returns.cov())


def cov_to_corr(cov):
    cov = sanitize_covariance_matrix(cov)
    if cov.empty:
        return pd.DataFrame()
    vol = np.sqrt(np.diag(cov.values))
    denom = np.outer(vol, vol)
    corr = cov.values / np.where(denom <= 0, np.nan, denom)
    corr = pd.DataFrame(corr, index=cov.index, columns=cov.columns)
    corr = corr.replace([np.inf, -np.inf], np.nan).clip(-1.0, 1.0)
    np.fill_diagonal(corr.values, 1.0)
    return corr


def hierarchical_fallback_weights(cov):
    clean_cov = sanitize_covariance_matrix(cov)
    if clean_cov.empty:
        return pd.Series(dtype=float)
    if clean_cov.shape[0] == 1:
        return pd.Series([1.0], index=clean_cov.index)
    return inverse_vol_weights_from_cov(clean_cov)


def prepare_hierarchical_inputs(cov, min_assets=2):
    clean_cov = sanitize_covariance_matrix(cov)
    diagnostics = {
        "hierarchical_fallback": False,
        "hierarchical_reason": "",
        "hierarchical_valid_sleeves": int(clean_cov.shape[0]),
    }
    if clean_cov.shape[0] < min_assets:
        diagnostics.update({"hierarchical_fallback": True, "hierarchical_reason": "too_few_valid_sleeves"})
        return clean_cov, pd.DataFrame(), None, diagnostics
    corr = cov_to_corr(clean_cov)
    if corr.empty or not np.isfinite(corr.values).all():
        valid = corr.index[np.isfinite(corr).all(axis=1)].tolist() if not corr.empty else []
        if len(valid) >= min_assets:
            clean_cov = clean_cov.loc[valid, valid]
            corr = cov_to_corr(clean_cov)
        if corr.empty or not np.isfinite(corr.values).all():
            diagnostics.update({
                "hierarchical_fallback": True,
                "hierarchical_reason": "non_finite_correlation",
                "hierarchical_valid_sleeves": int(clean_cov.shape[0]),
            })
            return clean_cov, corr, None, diagnostics
    dist_values = np.sqrt(np.clip((1.0 - corr.values) / 2.0, 0.0, 1.0))
    np.fill_diagonal(dist_values, 0.0)
    if not np.isfinite(dist_values).all():
        diagnostics.update({
            "hierarchical_fallback": True,
            "hierarchical_reason": "non_finite_distance_matrix",
            "hierarchical_valid_sleeves": int(clean_cov.shape[0]),
        })
        return clean_cov, corr, None, diagnostics
    condensed = squareform(dist_values, checks=False)
    if condensed.size == 0 or not np.isfinite(condensed).all():
        diagnostics.update({
            "hierarchical_fallback": True,
            "hierarchical_reason": "invalid_condensed_distance",
            "hierarchical_valid_sleeves": int(clean_cov.shape[0]),
        })
        return clean_cov, corr, None, diagnostics
    diagnostics["hierarchical_valid_sleeves"] = int(clean_cov.shape[0])
    return clean_cov, corr, condensed, diagnostics


def inverse_vol_weights_from_cov(cov):
    vol = pd.Series(np.sqrt(np.diag(cov.values)), index=cov.index)
    inv = safe_inverse(vol)
    return normalize_long_only(inv, max_weight=1.0)


def cluster_variance(cov, members):
    subcov = cov.loc[members, members]
    weights = inverse_vol_weights_from_cov(subcov)
    return float(weights.values @ subcov.values @ weights.values)


def solve_slsqp(objective, x0, bounds, constraints):
    if minimize is None:
        return None
    try:
        result = minimize(objective, x0=x0, method="SLSQP", bounds=bounds, constraints=constraints, options={"maxiter": 500, "ftol": 1e-9, "disp": False})
        if result.success:
            return result.x
    except Exception:
        return None
    return None


def optimize_min_variance(cov, prev_weights=None):
    names = list(cov.index)
    base = inverse_vol_weights_from_cov(cov).reindex(names)
    if len(names) == 1:
        return pd.Series([1.0], index=names)
    prev = base if prev_weights is None else normalize_long_only(prev_weights.reindex(names).fillna(0.0), max_weight=MAX_SLEEVE_WEIGHT)
    bounds = [(0.0, MAX_SLEEVE_WEIGHT)] * len(names)
    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    def objective(w):
        risk = float(w @ cov.values @ w)
        penalty = TURNOVER_PENALTY * float(np.sum((w - prev.values) ** 2))
        return risk + penalty
    solution = solve_slsqp(objective, x0=base.values, bounds=bounds, constraints=constraints)
    if solution is None:
        return normalize_long_only(base, max_weight=MAX_SLEEVE_WEIGHT)
    return normalize_long_only(pd.Series(solution, index=names), max_weight=MAX_SLEEVE_WEIGHT)


def optimize_mean_variance(mu, cov, prev_weights=None):
    names = list(cov.index)
    base = inverse_vol_weights_from_cov(cov).reindex(names)
    prev = base if prev_weights is None else normalize_long_only(prev_weights.reindex(names).fillna(0.0), max_weight=MAX_SLEEVE_WEIGHT)
    mu = pd.Series(mu, index=names).fillna(0.0)
    bounds = [(0.0, MAX_SLEEVE_WEIGHT)] * len(names)
    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    def objective(w):
        reward = float(mu.values @ w)
        risk = float(w @ cov.values @ w)
        penalty = TURNOVER_PENALTY * float(np.sum((w - prev.values) ** 2))
        return -(reward - 0.5 * RISK_AVERSION * risk) + penalty
    solution = solve_slsqp(objective, x0=base.values, bounds=bounds, constraints=constraints)
    if solution is None:
        return normalize_long_only(base, max_weight=MAX_SLEEVE_WEIGHT)
    return normalize_long_only(pd.Series(solution, index=names), max_weight=MAX_SLEEVE_WEIGHT)


def optimize_max_sharpe(mu, cov, prev_weights=None):
    names = list(cov.index)
    base = inverse_vol_weights_from_cov(cov).reindex(names)
    prev = base if prev_weights is None else normalize_long_only(prev_weights.reindex(names).fillna(0.0), max_weight=MAX_SLEEVE_WEIGHT)
    mu = pd.Series(mu, index=names).fillna(0.0)
    bounds = [(0.0, MAX_SLEEVE_WEIGHT)] * len(names)
    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    def objective(w):
        vol = np.sqrt(max(float(w @ cov.values @ w), 1e-12))
        sharpe = float(mu.values @ w) / vol
        penalty = TURNOVER_PENALTY * float(np.sum((w - prev.values) ** 2))
        return -sharpe + penalty
    solution = solve_slsqp(objective, x0=base.values, bounds=bounds, constraints=constraints)
    if solution is None:
        return normalize_long_only(base, max_weight=MAX_SLEEVE_WEIGHT)
    return normalize_long_only(pd.Series(solution, index=names), max_weight=MAX_SLEEVE_WEIGHT)


def optimize_max_diversification(cov, prev_weights=None):
    names = list(cov.index)
    base = inverse_vol_weights_from_cov(cov).reindex(names)
    prev = base if prev_weights is None else normalize_long_only(prev_weights.reindex(names).fillna(0.0), max_weight=MAX_SLEEVE_WEIGHT)
    vol = pd.Series(np.sqrt(np.diag(cov.values)), index=names)
    bounds = [(0.0, MAX_SLEEVE_WEIGHT)] * len(names)
    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    def objective(w):
        portfolio_vol = np.sqrt(max(float(w @ cov.values @ w), 1e-12))
        div_ratio = float(vol.values @ w) / portfolio_vol
        penalty = TURNOVER_PENALTY * float(np.sum((w - prev.values) ** 2))
        return -div_ratio + penalty
    solution = solve_slsqp(objective, x0=base.values, bounds=bounds, constraints=constraints)
    if solution is None:
        return normalize_long_only(base, max_weight=MAX_SLEEVE_WEIGHT)
    return normalize_long_only(pd.Series(solution, index=names), max_weight=MAX_SLEEVE_WEIGHT)


def optimize_erc(cov, prev_weights=None):
    names = list(cov.index)
    base = inverse_vol_weights_from_cov(cov).reindex(names)
    prev = base if prev_weights is None else normalize_long_only(prev_weights.reindex(names).fillna(0.0), max_weight=MAX_SLEEVE_WEIGHT)
    bounds = [(0.0, MAX_SLEEVE_WEIGHT)] * len(names)
    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    def objective(w):
        port_var = max(float(w @ cov.values @ w), 1e-12)
        marginal = cov.values @ w
        rc = w * marginal / np.sqrt(port_var)
        target = rc.mean()
        erc_loss = float(np.sum((rc - target) ** 2))
        penalty = TURNOVER_PENALTY * float(np.sum((w - prev.values) ** 2))
        return erc_loss + penalty
    solution = solve_slsqp(objective, x0=base.values, bounds=bounds, constraints=constraints)
    if solution is None:
        return normalize_long_only(base, max_weight=MAX_SLEEVE_WEIGHT)
    return normalize_long_only(pd.Series(solution, index=names), max_weight=MAX_SLEEVE_WEIGHT)


def optimize_hrp(cov, return_diagnostics=False):
    fallback = hierarchical_fallback_weights(cov)
    diagnostics = {"hierarchical_fallback": False, "hierarchical_reason": "", "hierarchical_valid_sleeves": int(len(fallback))}
    if linkage is None or squareform is None or leaves_list is None:
        diagnostics.update({"hierarchical_fallback": True, "hierarchical_reason": "scipy_unavailable"})
        result = fallback
    else:
        clean_cov, corr, condensed, diagnostics = prepare_hierarchical_inputs(cov, min_assets=2)
        if condensed is None or clean_cov.shape[0] < 2:
            result = hierarchical_fallback_weights(clean_cov if not clean_cov.empty else cov)
        else:
            try:
                link = linkage(condensed, method="single")
                ordered_names = corr.index[leaves_list(link)].tolist()
                weights = pd.Series(1.0, index=ordered_names)
                clusters = [ordered_names]
                while clusters:
                    cluster = clusters.pop(0)
                    if len(cluster) <= 1:
                        continue
                    split = len(cluster) // 2
                    left = cluster[:split]
                    right = cluster[split:]
                    left_var = cluster_variance(clean_cov, left)
                    right_var = cluster_variance(clean_cov, right)
                    alpha = 1.0 - left_var / max(left_var + right_var, 1e-12)
                    weights[left] *= alpha
                    weights[right] *= 1.0 - alpha
                    clusters.extend([left, right])
                result = normalize_long_only(weights.reindex(clean_cov.index).fillna(0.0), max_weight=MAX_SLEEVE_WEIGHT)
            except Exception:
                diagnostics.update({"hierarchical_fallback": True, "hierarchical_reason": "clustering_error"})
                result = hierarchical_fallback_weights(clean_cov)
    if return_diagnostics:
        return result, diagnostics
    return result


def optimize_herc(cov, return_diagnostics=False):
    if linkage is None or squareform is None or fcluster is None:
        result, diagnostics = optimize_hrp(cov, return_diagnostics=True)
        diagnostics.update({"hierarchical_fallback": True, "hierarchical_reason": "scipy_unavailable"})
        if return_diagnostics:
            return result, diagnostics
        return result
    clean_cov, corr, condensed, diagnostics = prepare_hierarchical_inputs(cov, min_assets=2)
    if condensed is None or clean_cov.shape[0] < 2:
        result = hierarchical_fallback_weights(clean_cov if not clean_cov.empty else cov)
        if return_diagnostics:
            return result, diagnostics
        return result
    try:
        # Ward linkage assumes Euclidean feature-space variance; average linkage is safer for a precomputed correlation-distance matrix.
        link = linkage(condensed, method="average")
        cluster_count = min(max(2, int(np.ceil(np.sqrt(len(clean_cov))))), len(clean_cov))
        labels = pd.Series(fcluster(link, cluster_count, criterion="maxclust"), index=clean_cov.index)
        cluster_portfolios = {}
        for cluster_id in sorted(labels.unique()):
            members = labels[labels == cluster_id].index.tolist()
            subcov = clean_cov.loc[members, members]
            cluster_portfolios[cluster_id] = inverse_vol_weights_from_cov(subcov)
        cluster_names = [f"cluster_{cid}" for cid in sorted(cluster_portfolios)]
        cluster_cov = pd.DataFrame(index=cluster_names, columns=cluster_names, dtype=float)
        for left_id, left_name in zip(sorted(cluster_portfolios), cluster_names):
            for right_id, right_name in zip(sorted(cluster_portfolios), cluster_names):
                w_left = cluster_portfolios[left_id].reindex(clean_cov.index).fillna(0.0).values
                w_right = cluster_portfolios[right_id].reindex(clean_cov.index).fillna(0.0).values
                cluster_cov.loc[left_name, right_name] = float(w_left @ clean_cov.values @ w_right)
        cluster_cov = sanitize_covariance_matrix(cluster_cov)
        if cluster_cov.empty:
            diagnostics.update({"hierarchical_fallback": True, "hierarchical_reason": "empty_cluster_covariance"})
            result = hierarchical_fallback_weights(clean_cov)
        elif cluster_cov.shape[0] == 1:
            only_cluster_id = next(iter(cluster_portfolios))
            result = normalize_long_only(cluster_portfolios[only_cluster_id].reindex(clean_cov.index).fillna(0.0), max_weight=MAX_SLEEVE_WEIGHT)
        else:
            cluster_alloc = optimize_erc(cluster_cov, prev_weights=None)
            final = pd.Series(0.0, index=clean_cov.index)
            for cluster_id, cluster_name in zip(sorted(cluster_portfolios), cluster_names):
                if cluster_name not in cluster_alloc.index:
                    continue
                final.loc[cluster_portfolios[cluster_id].index] += cluster_alloc[cluster_name] * cluster_portfolios[cluster_id]
            result = normalize_long_only(final, max_weight=MAX_SLEEVE_WEIGHT)
    except Exception:
        diagnostics.update({"hierarchical_fallback": True, "hierarchical_reason": "clustering_error"})
        result = hierarchical_fallback_weights(clean_cov)
    if return_diagnostics:
        return result, diagnostics
    return result


def black_litterman_confidence_from_spread(
    spread,
    floor=BL_CONFIDENCE_FLOOR,
    ceiling=BL_CONFIDENCE_CEIL,
    spread_divisor=BL_CONFIDENCE_SPREAD_DIVISOR,
):
    """Map a relative-view score spread into a practical BL confidence value."""
    return float(np.clip(abs(spread) / max(spread_divisor, 1e-12), floor, ceiling))


def black_litterman_posterior(cov, score_row, prior_weights=None):
    names = list(cov.index)
    prior = pd.Series(1.0 / len(names), index=names) if prior_weights is None else normalize_long_only(prior_weights.reindex(names).fillna(0.0), max_weight=1.0)
    pi = BL_RISK_AVERSION * (cov.values @ prior.values)
    scores = pd.Series(score_row, index=names).dropna().sort_values()
    diagnostics = {"view_count": 0, "view_confidence": np.nan, "view_spread": np.nan}
    if len(scores) < 4:
        return pd.Series(pi, index=names), diagnostics
    bottom = scores.head(max(1, len(scores) // 3))
    top = scores.tail(max(1, len(scores) // 3))
    p = pd.Series(0.0, index=names)
    p.loc[top.index] = 1.0 / len(top)
    p.loc[bottom.index] = -1.0 / len(bottom)
    spread = float(top.mean() - bottom.mean())
    # Heuristic Idzorek-style mapping: useful for calibration, but not theoretically unique.
    confidence = black_litterman_confidence_from_spread(spread)
    q = np.array([spread * (MEAN_RETURN_SCALE_ANN / WEEKS_PER_YEAR)])
    P = p.values.reshape(1, -1)
    tau_cov = BL_TAU * cov.values
    omega = np.diag(np.diag(P @ tau_cov @ P.T) / confidence)
    middle = np.linalg.pinv(P @ tau_cov @ P.T + omega)
    posterior = pi + tau_cov @ P.T @ middle @ (q - P @ pi)
    diagnostics.update({"view_count": 1, "view_confidence": confidence, "view_spread": spread})
    return pd.Series(posterior.ravel(), index=names), diagnostics


def optimize_cvar(train_returns, prev_weights=None):
    if cp is None or train_returns.empty:
        return inverse_vol_weights_from_cov(train_returns.cov()) if not train_returns.empty else pd.Series(dtype=float)
    names = list(train_returns.columns)
    returns = train_returns[names].dropna(how="any")
    if returns.empty:
        return inverse_vol_weights_from_cov(train_returns.cov())
    w = cp.Variable(len(names))
    alpha = cp.Variable()
    u = cp.Variable(len(returns))
    losses = -returns.values @ w
    constraints = [
        cp.sum(w) == 1.0,
        w >= 0.0,
        w <= MAX_SLEEVE_WEIGHT,
        u >= 0.0,
        u >= losses - alpha,
    ]
    objective = cp.Minimize(alpha + (1.0 / ((1.0 - CVAR_LEVEL) * len(returns))) * cp.sum(u))
    problem = cp.Problem(objective, constraints)
    try:
        problem.solve(solver=cp.SCS, verbose=False)
    except Exception:
        return inverse_vol_weights_from_cov(train_returns.cov())
    if w.value is None:
        return inverse_vol_weights_from_cov(train_returns.cov())
    return normalize_long_only(pd.Series(np.asarray(w.value).ravel(), index=names), max_weight=MAX_SLEEVE_WEIGHT)


def select_active_sleeves(train_returns, min_obs=MIN_TRAIN_OBS):
    counts = train_returns.count()
    return counts[counts >= min_obs].index.tolist()


def apply_overlays(
    raw_weights,
    cov,
    regime_row,
    prev_weights=None,
    target_vol_ceil=TARGET_VOL_CEIL,
    sleeve_reallocation_speed=SLEEVE_REALLOCATION_SPEED,
):
    raw_weights = normalize_long_only(raw_weights, max_weight=MAX_SLEEVE_WEIGHT)
    if prev_weights is not None and not prev_weights.empty:
        prev_weights = normalize_long_only(prev_weights.reindex(raw_weights.index).fillna(0.0), max_weight=MAX_SLEEVE_WEIGHT)
        blended = (1.0 - sleeve_reallocation_speed) * prev_weights + sleeve_reallocation_speed * raw_weights
    else:
        blended = raw_weights.copy()
    blended = normalize_long_only(blended, max_weight=MAX_SLEEVE_WEIGHT)
    predicted_ann_vol = np.sqrt(max(float(blended.values @ cov.values @ blended.values), 0.0)) * np.sqrt(WEEKS_PER_YEAR)
    # Layer 2B's market-vol scaler is a broad regime proxy; this is the final portfolio-aware target-vol cap, so it stays stricter and never scales above full risky exposure.
    target_vol_multiplier = 1.0 if predicted_ann_vol <= 0 or pd.isna(predicted_ann_vol) else float(np.clip(TARGET_VOL_ANN / predicted_ann_vol, TARGET_VOL_FLOOR, target_vol_ceil))
    regime_multiplier = float(regime_row.get("overlay_multiplier", 1.0)) if isinstance(regime_row, pd.Series) else 1.0
    gross_multiplier = float(min(1.0, regime_multiplier, target_vol_multiplier))
    risky_weights = blended * gross_multiplier
    cash_weight = max(0.0, 1.0 - risky_weights.sum())
    diagnostics = {
        "predicted_ann_vol": predicted_ann_vol,
        "target_vol_multiplier": target_vol_multiplier,
        "regime_multiplier": regime_multiplier,
        "gross_multiplier": gross_multiplier,
        "cash_weight": cash_weight,
    }
    return risky_weights, cash_weight, diagnostics


def apply_etf_cap(final_weights, cash_proxy):
    final_weights = pd.Series(final_weights, dtype=float).clip(lower=0.0).fillna(0.0)
    risky = final_weights.drop(labels=[cash_proxy], errors="ignore").clip(upper=MAX_ETF_WEIGHT)
    capped = risky.copy()
    capped = capped / max(capped.sum(), 1.0) if capped.sum() > 1.0 else capped
    capped_cash = 1.0 - capped.sum()
    result = capped.reindex(list(final_weights.index), fill_value=0.0)
    if cash_proxy not in result.index:
        result.loc[cash_proxy] = 0.0
    result.loc[cash_proxy] = capped_cash
    return result


def build_lookthrough_etf_weights(date, sleeve_weights, sleeve_positions, universe_columns, cash_proxy, cash_weight):
    final = pd.Series(0.0, index=list(universe_columns), dtype=float)
    if cash_proxy not in final.index:
        final.loc[cash_proxy] = 0.0
    for sleeve_name, sleeve_weight in pd.Series(sleeve_weights).dropna().items():
        if sleeve_name not in sleeve_positions:
            continue
        panel = sleeve_positions[sleeve_name]
        if date not in panel.index:
            continue
        row = panel.loc[date].reindex(final.index).fillna(0.0)
        final = final.add(sleeve_weight * row, fill_value=0.0)
    final.loc[cash_proxy] = final.get(cash_proxy, 0.0) + cash_weight
    return apply_etf_cap(final, cash_proxy=cash_proxy)


def run_method_backtest(
    method_name,
    method_category,
    engine,
    sleeve_return_panel,
    sleeve_positions,
    forward_weekly_returns,
    conviction_inputs,
    regime_states,
    cash_proxy,
    train_window_weeks=TRAIN_WINDOW_WEEKS,
    expected_return_key=None,
    covariance_method="ledoit_wolf",
    target_vol_ceil=TARGET_VOL_CEIL,
    sleeve_reallocation_speed=SLEEVE_REALLOCATION_SPEED,
):
    all_dates = sleeve_return_panel.index
    rebalance_dates = rebalance_mask(all_dates, REBALANCE_FREQUENCY)
    sleeve_names = list(sleeve_return_panel.columns)
    current_risky_alloc = pd.Series(0.0, index=sleeve_names, dtype=float)
    current_cash_weight = 1.0
    sleeve_alloc_rows = []
    etf_weight_rows = []
    diag_rows = []
    # The return panel is already shifted to the next holding period; name it explicitly to avoid future timing mistakes.
    lookthrough_columns = list(forward_weekly_returns.columns)
    for date in all_dates:
        if rebalance_dates.loc[date]:
            train_slice = sleeve_return_panel.loc[:date].tail(train_window_weeks)
            active = select_active_sleeves(train_slice)
            if len(active) >= 2:
                train = train_slice[active].dropna(how="any")
                if len(train) >= max(26, min(MIN_TRAIN_OBS, train_window_weeks // 2)):
                    cov = estimate_covariance(train, method=covariance_method)
                    if cov.empty:
                        continue
                    active = list(cov.index)
                    train = train.reindex(columns=active).dropna(how="any")
                    prev_active = current_risky_alloc.reindex(active).fillna(0.0)
                    mu = pd.Series(0.0, index=active)
                    bl_diag = {"view_count": 0, "view_confidence": np.nan, "view_spread": np.nan}
                    hier_diag = {"hierarchical_fallback": False, "hierarchical_reason": "", "hierarchical_valid_sleeves": len(active)}
                    if expected_return_key is not None and expected_return_key in conviction_inputs:
                        score_row = conviction_inputs[expected_return_key].reindex(columns=sleeve_names).loc[date].reindex(active)
                        mu = score_row_to_weekly_mu(score_row, train)
                    if engine == "equal_weight":
                        raw = pd.Series(1.0 / len(active), index=active)
                    elif engine == "inverse_vol":
                        raw = inverse_vol_weights_from_cov(cov)
                    elif engine == "min_variance":
                        raw = optimize_min_variance(cov, prev_weights=prev_active)
                    elif engine == "mvo":
                        raw = optimize_mean_variance(mu, cov, prev_weights=prev_active)
                    elif engine == "max_sharpe":
                        raw = optimize_max_sharpe(mu, cov, prev_weights=prev_active)
                    elif engine == "black_litterman":
                        posterior_mu, bl_diag = black_litterman_posterior(cov, conviction_inputs[expected_return_key].loc[date].reindex(active), prior_weights=prev_active if prev_active.sum() > 0 else None)
                        raw = optimize_mean_variance(posterior_mu, cov, prev_weights=prev_active)
                    elif engine == "erc":
                        raw = optimize_erc(cov, prev_weights=prev_active)
                    elif engine == "hrp":
                        # Some train windows produce non-finite correlation distances after missing-data or zero-variance cleanup; validate first and fall back to inverse vol when clustering is not well-posed.
                        raw, hier_diag = optimize_hrp(cov, return_diagnostics=True)
                    elif engine == "herc":
                        raw, hier_diag = optimize_herc(cov, return_diagnostics=True)
                    elif engine == "max_diversification":
                        raw = optimize_max_diversification(cov, prev_weights=prev_active)
                    elif engine == "cvar":
                        raw = optimize_cvar(train, prev_weights=prev_active)
                    else:
                        raise ValueError(f"Unknown engine: {engine}")
                    overlay_row = regime_states.loc[date] if date in regime_states.index else pd.Series(dtype=float)
                    risky_weights, cash_weight, overlay_diag = apply_overlays(
                        raw,
                        cov,
                        overlay_row,
                        prev_weights=prev_active,
                        target_vol_ceil=target_vol_ceil,
                        sleeve_reallocation_speed=sleeve_reallocation_speed,
                    )
                    current_risky_alloc = pd.Series(0.0, index=sleeve_names, dtype=float)
                    current_risky_alloc.loc[risky_weights.index] = risky_weights
                    current_cash_weight = cash_weight
                    diag_rows.append(
                        {
                            "Date": date,
                            "method_name": method_name,
                            "engine": engine,
                            "method_category": method_category,
                            "active_sleeves": len(active),
                            "expected_return_key": expected_return_key or "n/a",
                            "covariance_method": covariance_method,
                            **overlay_diag,
                            **bl_diag,
                            **hier_diag,
                        }
                    )
        allocation_row = current_risky_alloc.copy()
        allocation_row.loc[f"cash::{cash_proxy}"] = current_cash_weight
        allocation_row.name = date
        sleeve_alloc_rows.append(allocation_row)
        etf_row = build_lookthrough_etf_weights(
            date=date,
            sleeve_weights=current_risky_alloc,
            sleeve_positions=sleeve_positions,
            universe_columns=lookthrough_columns,
            cash_proxy=cash_proxy,
            cash_weight=current_cash_weight,
        )
        etf_row.name = date
        etf_weight_rows.append(etf_row)
    sleeve_alloc = pd.DataFrame(sleeve_alloc_rows).sort_index().fillna(0.0)
    etf_weights = pd.DataFrame(etf_weight_rows).sort_index().fillna(0.0)
    path = compute_portfolio_path(etf_weights, forward_weekly_returns.reindex(index=etf_weights.index, columns=etf_weights.columns), transaction_cost_bps=DEFAULT_COST_BPS)
    diagnostics = pd.DataFrame(diag_rows)
    return sleeve_alloc, etf_weights, path, diagnostics


def static_in_sample_weights(engine, sleeve_return_panel, conviction_inputs, expected_return_key=None, covariance_method="ledoit_wolf"):
    train = sanitize_train_window(sleeve_return_panel, min_obs=min(10, max(len(sleeve_return_panel), 1)))
    cov = estimate_covariance(train, method=covariance_method)
    if cov.empty:
        return pd.Series(dtype=float)
    active = list(cov.index)
    train = train.reindex(columns=active).dropna(how="any")
    mu = pd.Series(0.0, index=active)
    if expected_return_key is not None and expected_return_key in conviction_inputs:
        avg_score = conviction_inputs[expected_return_key].reindex(columns=active).mean(axis=0)
        mu = score_row_to_weekly_mu(avg_score, train)
    if engine == "equal_weight":
        return pd.Series(1.0 / len(active), index=active)
    if engine == "inverse_vol":
        return inverse_vol_weights_from_cov(cov)
    if engine == "min_variance":
        return optimize_min_variance(cov)
    if engine == "mvo":
        return optimize_mean_variance(mu, cov)
    if engine == "max_sharpe":
        return optimize_max_sharpe(mu, cov)
    if engine == "black_litterman":
        posterior_mu, _ = black_litterman_posterior(cov, conviction_inputs[expected_return_key].mean(axis=0).reindex(active))
        return optimize_mean_variance(posterior_mu, cov)
    if engine == "erc":
        return optimize_erc(cov)
    if engine == "hrp":
        return optimize_hrp(cov)
    if engine == "herc":
        return optimize_herc(cov)
    if engine == "max_diversification":
        return optimize_max_diversification(cov)
    if engine == "cvar":
        return optimize_cvar(train)
    raise ValueError(f"Unknown engine: {engine}")


## 3. Load Existing Layer 1 / 2 Outputs

This notebook intentionally prefers the saved artifacts from the previous layers.

**From Layer 1**

- tradable signal panels
- signal manifest

**From Layer 2A**

- strategy position histories
- strategy return histories
- strategy summary table
- rolling signal weight history for the IC-weighted and regime-conditioned composites

**From Layer 2B**

- regime states
- overlay multipliers
- covariance snapshots

The notebook raises a clear error if the upstream chain has not been run yet, because portfolio construction without the earlier layers would only recreate assumptions that the pipeline is supposed to make explicit.


In [ ]:
data_hub_dir, missing_hub = choose_input_dir(DATA_HUB_CANDIDATES, REQUIRED_DATA_HUB)
layer1_dir, missing_layer1 = choose_input_dir(LAYER1_CANDIDATES, REQUIRED_LAYER1)
layer2a_dir, missing_layer2a = choose_input_dir(LAYER2A_CANDIDATES, REQUIRED_LAYER2A)
layer2b_dir, missing_layer2b = choose_input_dir(LAYER2B_CANDIDATES, REQUIRED_LAYER2B)

if missing_hub:
    raise FileNotFoundError(f"Missing required data-hub inputs: {missing_hub}. Run 01_data_hub.ipynb first.")
if missing_layer1:
    raise FileNotFoundError(f"Missing required Layer 1 inputs: {missing_layer1}. Run 02_layer1_alpha_signals.ipynb first.")
if missing_layer2a:
    raise FileNotFoundError(f"Missing required Layer 2A inputs: {missing_layer2a}. Run 03_layer2a_strategy_logic.ipynb first.")
if missing_layer2b:
    raise FileNotFoundError(f"Missing required Layer 2B inputs: {missing_layer2b}. Run 04_layer2b_risk_regime_engine.ipynb first.")

weekly_prices = read_panel_csv(data_hub_dir / "weekly_prices.csv")
weekly_log_returns = read_panel_csv(data_hub_dir / "weekly_returns.csv")
weekly_simple_returns = np.expm1(weekly_log_returns).reindex_like(weekly_prices)
next_week_returns = weekly_simple_returns.shift(-1)
daily_log_returns = read_panel_csv(data_hub_dir / "daily_returns.csv")
universe_def = read_json_file(data_hub_dir / "universe.json")
proxy_mapping = read_json_file(data_hub_dir / "proxy_mapping.json")
market_proxy_frame = read_panel_csv(data_hub_dir / "market_proxy_weekly.csv")

signal_manifest = read_json_file(layer1_dir / "signal_manifest.json")
strategy_summary = read_table_csv(layer2a_dir / "strategy_summary_table.csv")
strategy_weight_history_long = read_table_csv(layer2a_dir / "strategy_signal_weight_history.csv")
regime_states = read_panel_csv(layer2b_dir / "regime_states.csv")
regime_score = read_panel_csv(layer2b_dir / "regime_score.csv")
covariance_long = read_table_csv(layer2b_dir / "covariance_snapshots_long.csv")
covariance_lw_long = read_table_csv(layer2b_dir / "covariance_ledoit_wolf_snapshots_long.csv")

cash_proxy = get_proxy_ticker(proxy_mapping, "cash_proxy", fallback="SHY")
duration_proxy = get_proxy_ticker(proxy_mapping, "duration_proxy", fallback="IEF")
market_proxy = get_proxy_ticker(proxy_mapping, "market_proxy", fallback="SPY")

available_sleeves = []
sleeve_positions = {}
sleeve_returns = {}
for sleeve_name, sleeve_role in CANDIDATE_SLEEVES:
    pos_path = layer2a_dir / f"strategy_positions_{sleeve_name}.csv"
    ret_path = layer2a_dir / f"strategy_returns_{sleeve_name}.csv"
    positions = read_strategy_positions(pos_path)
    rets = read_strategy_returns(ret_path)
    if positions.empty or rets.empty:
        continue
    positions = positions.reindex(index=weekly_prices.index).ffill().fillna(0.0)
    rets = rets.reindex(index=weekly_prices.index)
    sleeve_positions[sleeve_name] = positions
    sleeve_returns[sleeve_name] = rets["net_return"] if "net_return" in rets.columns else rets.iloc[:, 0]
    available_sleeves.append({"sleeve_name": sleeve_name, "role": sleeve_role})

sleeve_universe = pd.DataFrame(available_sleeves)
if len(sleeve_positions) < 4:
    raise ValueError(
        "Too few Layer 2 sleeves are available to compare portfolio construction methods. "
        "Run 03_layer2a_strategy_logic.ipynb and ensure the core sleeves were saved."
    )

sleeve_return_panel = pd.DataFrame(sleeve_returns).reindex(weekly_prices.index)
sleeve_return_panel.index.name = "Date"

signal_panels = {}
for signal_name, (file_name, value_col) in SIGNAL_COLUMN_MAP.items():
    long_df = read_signal_long(layer1_dir / file_name)
    panel = long_signal_to_panel(long_df, value_col=value_col, index=weekly_prices.index, columns=weekly_prices.columns)
    if panel.notna().sum().sum() > 0:
        signal_panels[signal_name] = panel

if len(signal_panels) < 4:
    raise ValueError(
        "Too few tradable Layer 1 panels were loaded to form stable conviction inputs. "
        "Check that 02_layer1_alpha_signals.ipynb saved the expected signal CSVs."
    )

print("Data hub directory:", data_hub_dir.resolve())
print("Layer 1 directory:", layer1_dir.resolve())
print("Layer 2A directory:", layer2a_dir.resolve())
print("Layer 2B directory:", layer2b_dir.resolve())
print("Available sleeves:", list(sleeve_positions.keys()))
print("Loaded signal panels:", list(signal_panels.keys()))


## 4. Candidate Conviction Inputs and Why They Matter

Portfolio construction needs some notion of **expected return**, **view strength**, or **conviction**.

That is where many optimizers become fragile. If expected returns are noisy, classical mean-variance methods can look brilliant in sample and unstable out of sample.

To avoid that trap, the notebook supports several simple, transparent conviction inputs:

1. **Equal-weight Layer 1 composite**
   A robust baseline that avoids unstable signal weighting.

2. **IC-weighted Layer 1 composite**
   Uses Layer 2A's rolling signal weights, currently built on a fixed tactical 4-week IC horizon with shrinkage to equal weight.

3. **Regime-conditioned Layer 1 composite**
   Uses the regime-aware signal weighting already defined in Layer 2A.

4. **Layer 2 sleeve momentum**
   A small overlay based on trailing sleeve-level performance, included as a practical strategy-level score rather than as a new alpha claim.

These are still only **expected-return proxies**, not forecasts we should believe literally. We intentionally transform them into **small, shrunk weekly expected returns** before feeding them into mean-sensitive allocators.


In [ ]:
baseline_signal_names = [name for name in [
    "xsmom_global",
    "multi_mom_invvol",
    "residual_momentum",
    "quality_proxy",
    "value_proxy",
    "carry_proxy",
    "bab_proxy",
    "reversal_4w_global",
] if name in signal_panels]

baseline_signal_panels = {name: signal_panels[name] for name in baseline_signal_names}

equal_signal_composite_asset = combine_signal_panels(baseline_signal_panels, weight_history=None)

ic_weight_history = weight_history_from_long(
    strategy_weight_history_long,
    strategy_name=EXPECTED_RETURN_HISTORY_NAMES["ic_signal"],
    dates=weekly_prices.index,
    signal_names=baseline_signal_names,
)
regime_weight_history = weight_history_from_long(
    strategy_weight_history_long,
    strategy_name=EXPECTED_RETURN_HISTORY_NAMES["regime_signal"],
    dates=weekly_prices.index,
    signal_names=baseline_signal_names,
)

if ic_weight_history.notna().sum().sum() == 0:
    ic_weight_history = pd.DataFrame(1.0 / len(baseline_signal_names), index=weekly_prices.index, columns=baseline_signal_names)
if regime_weight_history.notna().sum().sum() == 0:
    regime_weight_history = pd.DataFrame(1.0 / len(baseline_signal_names), index=weekly_prices.index, columns=baseline_signal_names)

ic_signal_composite_asset = combine_signal_panels(baseline_signal_panels, weight_history=ic_weight_history)
regime_signal_composite_asset = combine_signal_panels(baseline_signal_panels, weight_history=regime_weight_history)

equal_signal_sleeve = sleeve_scores_from_asset_scores(equal_signal_composite_asset, sleeve_positions)
ic_signal_sleeve = sleeve_scores_from_asset_scores(ic_signal_composite_asset, sleeve_positions)
regime_signal_sleeve = sleeve_scores_from_asset_scores(regime_signal_composite_asset, sleeve_positions)

sleeve_momentum_raw = trailing_compound_return(sleeve_return_panel.fillna(0.0), lookback=13)
sleeve_momentum_score = row_zscore(sleeve_momentum_raw).clip(-2.0, 2.0)

common_sleeves = list(sleeve_positions.keys())
conviction_inputs = {
    "equal_signal": equal_signal_sleeve.reindex(columns=common_sleeves),
    "ic_signal": ic_signal_sleeve.reindex(columns=common_sleeves),
    "regime_signal": regime_signal_sleeve.reindex(columns=common_sleeves),
    "sleeve_momentum": sleeve_momentum_score.reindex(columns=common_sleeves),
}
conviction_inputs["default_blend"] = (
    0.35 * conviction_inputs["equal_signal"]
    + 0.35 * conviction_inputs["ic_signal"]
    + 0.20 * conviction_inputs["regime_signal"]
    + 0.10 * conviction_inputs["sleeve_momentum"]
)
sleeve_corr_context = mean_pairwise_correlation(sleeve_return_panel.fillna(0.0), window=26)

conviction_long = []
for input_name, panel in conviction_inputs.items():
    tidy = panel.copy()
    tidy["Date"] = tidy.index
    tidy = tidy.melt(id_vars="Date", var_name="sleeve_name", value_name="score")
    tidy["input_name"] = input_name
    conviction_long.append(tidy)
conviction_long = pd.concat(conviction_long, ignore_index=True)

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
conviction_inputs["default_blend"].plot(ax=axes[0], lw=1.1)
axes[0].set_title("Sleeve conviction inputs: default blended score")
sleeve_corr_context.plot(ax=axes[1], lw=1.4, color="darkorange")
axes[1].set_title("Sanity check: mean trailing pairwise sleeve correlation")
plt.tight_layout()
plt.show()

print("Candidate sleeves for Layer 3")
print(sleeve_universe.to_string(index=False))
print()
print("Conviction input availability")
print(conviction_long.groupby("input_name")["score"].apply(lambda s: s.notna().mean()).round(3).to_string())


## 5. Portfolio Construction Methods and Constraint Set

The comparison set below is intentionally split into categories:

**Baselines**

- Equal weight
- Inverse volatility

**Classical optimization**

- Minimum variance
- Mean-variance optimization
- Maximum Sharpe / tangency
- Black-Litterman

**Risk-based / hierarchical**

- Equal risk contribution (risk parity)
- Hierarchical risk parity (HRP)
- Hierarchical equal risk contribution (HERC)
- Maximum diversification

**Tail-risk-aware extension**

- CVaR minimization if `cvxpy` is available

All methods share the same practical constraints:

- long-only sleeve allocations
- sleeve weight cap
- ETF weight cap after look-through
- full investment via an explicit cash sleeve
- turnover-aware smoothing
- regime and volatility overlays

This is important: a constrained, slightly boring allocator is often more useful than an elegant unconstrained optimizer that produces unstable concentration.

A small consistency note: Layer 2B's `target_vol_multiplier` is a market-risk proxy used for regime conditioning and can stay slightly looser. Layer 3 applies the final portfolio-aware target-vol cap, so its ceiling remains stricter at `1.00` and does not lever above full risky exposure.

Two implementation details are intentionally documented here. HERC uses average linkage because Ward linkage is designed for Euclidean feature-space variance, not arbitrary precomputed correlation distances. Black-Litterman uses a standard practical `tau` value, but the mapping from signal-spread strength to view confidence is heuristic calibration rather than a unique theoretical rule.


In [ ]:
method_specs = [
    {
        "method_name": "equal_weight",
        "category": "baseline",
        "engine": "equal_weight",
        "expected_return_key": None,
        "covariance_method": "sample",
        "description": "Equal allocation across the core Layer 2 sleeves.",
        "caveats": "Hard to beat out of sample; important baseline against optimizer overfitting.",
    },
    {
        "method_name": "inverse_vol",
        "category": "baseline",
        "engine": "inverse_vol",
        "expected_return_key": None,
        "covariance_method": "sample",
        "description": "Inverse-volatility allocation across sleeves.",
        "caveats": "Still simple, but implicitly leans into low-vol sleeves.",
    },
    {
        "method_name": "min_variance",
        "category": "classical_optimization",
        "engine": "min_variance",
        "expected_return_key": None,
        "covariance_method": "ledoit_wolf",
        "description": "Long-only minimum variance with shrinkage covariance and turnover smoothing.",
        "caveats": "Can become overly defensive when covariance estimates are noisy or highly clustered.",
    },
    {
        "method_name": "mvo_ic",
        "category": "classical_optimization",
        "engine": "mvo",
        "expected_return_key": "ic_signal",
        "covariance_method": "ledoit_wolf",
        "description": "Long-only mean-variance allocator using shrunk IC-weighted sleeve conviction inputs.",
        "caveats": "Expected-return sensitive by design; should be interpreted cautiously.",
    },
    {
        "method_name": "max_sharpe_ic",
        "category": "classical_optimization",
        "engine": "max_sharpe",
        "expected_return_key": "ic_signal",
        "covariance_method": "ledoit_wolf",
        "description": "Constrained tangency-style allocator using shrunk IC-based expected returns.",
        "caveats": "Often looks strongest in sample and most fragile out of sample.",
    },
    {
        "method_name": "black_litterman_regime",
        "category": "classical_optimization",
        "engine": "black_litterman",
        "expected_return_key": "regime_signal",
        "covariance_method": "ledoit_wolf",
        "description": "Practical sleeve-level Black-Litterman using equal-weight prior and relative regime-aware views.",
        "caveats": "Adapted to strategy sleeves rather than market-cap assets; useful as a disciplined shrinkage framework, not magic.",
    },
    {
        "method_name": "erc_risk_parity",
        "category": "risk_based",
        "engine": "erc",
        "expected_return_key": None,
        "covariance_method": "ledoit_wolf",
        "description": "Long-only equal risk contribution allocation.",
        "caveats": "Robust when return estimates are noisy, but can overweight structurally low-vol sleeves.",
    },
    {
        "method_name": "hrp",
        "category": "hierarchical",
        "engine": "hrp",
        "expected_return_key": None,
        "covariance_method": "sample",
        "description": "Hierarchical Risk Parity using correlation clustering and recursive bisection.",
        "caveats": "More robust to inversion error, but still dependent on the correlation tree.",
    },
    {
        "method_name": "herc",
        "category": "hierarchical",
        "engine": "herc",
        "expected_return_key": None,
        "covariance_method": "sample",
        "description": "Hierarchical Equal Risk Contribution via cluster-level ERC and intra-cluster inverse vol.",
        "caveats": "With a small sleeve universe, differences versus HRP can be modest.",
    },
    {
        "method_name": "max_diversification",
        "category": "risk_based",
        "engine": "max_diversification",
        "expected_return_key": None,
        "covariance_method": "ledoit_wolf",
        "description": "Diversification-ratio allocator following the maximum-diversification literature.",
        "caveats": "Often close to inverse-vol or risk-parity behavior when correlations are stable.",
    },
]

if cp is not None:
    method_specs.append(
        {
            "method_name": "cvar_min",
            "category": "tail_risk_extension",
            "engine": "cvar",
            "expected_return_key": None,
            "covariance_method": "sample",
            "description": "CVaR-minimizing allocator using trailing sleeve return scenarios.",
            "caveats": "Useful tail-risk benchmark, but more computationally expensive and still data-hungry.",
        }
    )

method_frame = pd.DataFrame(method_specs)
print(method_frame[["method_name", "category", "engine", "expected_return_key", "covariance_method"]].to_string(index=False))


## 6. Walk-Forward Out-of-Sample Test

The main evaluation is a rolling out-of-sample comparison:

- **train window**: trailing multi-year history
- **rebalance**: monthly
- **evaluation frequency**: weekly returns
- **inputs at each rebalance**:
  - trailing sleeve return history
  - current tradable conviction inputs
  - current tradable regime overlay

Each method produces:

1. sleeve allocations
2. look-through ETF weights
3. realized gross / net returns
4. turnover and concentration diagnostics

This matters because a portfolio construction method that only looks good on *static weights* or *one set of hindsight parameters* is not yet useful for production.


In [ ]:
backtest_results = {}
comparison_rows = []
cost_rows = []
subperiod_rows = []
regime_rows = []
diagnostic_rows = []
manifest_records = []

for spec in method_specs:
    sleeve_alloc, etf_weights, path, diagnostics = run_method_backtest(
        method_name=spec["method_name"],
        method_category=spec["category"],
        engine=spec["engine"],
        sleeve_return_panel=sleeve_return_panel,
        sleeve_positions=sleeve_positions,
        forward_weekly_returns=next_week_returns.reindex(columns=weekly_prices.columns),
        conviction_inputs=conviction_inputs,
        regime_states=regime_states,
        cash_proxy=cash_proxy,
        train_window_weeks=TRAIN_WINDOW_WEEKS,
        expected_return_key=spec["expected_return_key"],
        covariance_method=spec["covariance_method"],
    )
    metrics = summary_metrics(
        path["net_return"],
        turnover_series=path["turnover"],
        weight_panel=etf_weights,
        allocation_panel=sleeve_alloc,
        trials=len(method_specs),
    )
    metrics.update(
        {
            "method_name": spec["method_name"],
            "category": spec["category"],
            "engine": spec["engine"],
            "expected_return_key": spec["expected_return_key"] or "n/a",
            "covariance_method": spec["covariance_method"],
            "description": spec["description"],
        }
    )
    comparison_rows.append(metrics)
    cost_rows.append(cost_sensitivity_table(etf_weights, next_week_returns.reindex(columns=etf_weights.columns), spec["method_name"]))
    subperiod_rows.append(subperiod_summary(spec["method_name"], path["net_return"]))
    regime_rows.append(regime_split_summary(spec["method_name"], path["net_return"], regime_states.get("risk_state", pd.Series(dtype=object))))
    if not diagnostics.empty:
        diagnostic_rows.append(diagnostics)
    backtest_results[spec["method_name"]] = {
        "spec": spec,
        "sleeve_alloc": sleeve_alloc,
        "weights": etf_weights,
        "path": path,
    }
    manifest_records.append(
        {
            "method_name": spec["method_name"],
            "category": spec["category"],
            "required_inputs": [
                "strategy_summary_table.csv",
                "strategy_positions_*.csv",
                "strategy_returns_*.csv",
                "regime_states.csv",
                "weekly_returns.csv",
            ],
            "constraints": {
                "long_only": True,
                "max_sleeve_weight": MAX_SLEEVE_WEIGHT,
                "max_etf_weight": MAX_ETF_WEIGHT,
                "cash_proxy": cash_proxy,
                "rebalance_frequency": REBALANCE_FREQUENCY,
            },
            "rebalance_frequency": REBALANCE_FREQUENCY,
            "risk_model_used": spec["covariance_method"],
            "expected_return_input_used": spec["expected_return_key"] or "none",
            "output_files": [
                f"portfolio_weights_{spec['method_name']}.csv",
                f"portfolio_sleeve_weights_{spec['method_name']}.csv",
                f"portfolio_returns_{spec['method_name']}.csv",
            ],
            "caveats": spec["caveats"],
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
cost_df = pd.concat(cost_rows, ignore_index=True) if cost_rows else pd.DataFrame()
subperiod_df = pd.concat(subperiod_rows, ignore_index=True) if subperiod_rows else pd.DataFrame()
regime_df = pd.concat(regime_rows, ignore_index=True) if regime_rows else pd.DataFrame()
diagnostic_df = pd.concat(diagnostic_rows, ignore_index=True) if diagnostic_rows else pd.DataFrame()

benchmark_rows = []
benchmark_names = [
    "baseline_market_proxy_buy_hold",
    "baseline_60_40_proxy",
    "dual_momentum_topn",
    "composite_regime_conditioned",
]
for benchmark_name in benchmark_names:
    returns_path = layer2a_dir / f"strategy_returns_{benchmark_name}.csv"
    returns_df = read_strategy_returns(returns_path)
    if returns_df.empty:
        continue
    benchmark_metrics = summary_metrics(returns_df.get("net_return", returns_df.iloc[:, 0]), trials=len(method_specs))
    benchmark_metrics.update({"method_name": benchmark_name, "category": "benchmark", "engine": "layer2_benchmark"})
    benchmark_rows.append(benchmark_metrics)
benchmark_df = pd.DataFrame(benchmark_rows)

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
to_plot = comparison_df.sort_values("sharpe", ascending=False)["method_name"].head(4).tolist()
plotted = []
for name in to_plot:
    if name in backtest_results:
        backtest_results[name]["path"]["wealth"].plot(ax=axes[0], lw=1.8, label=name)
        backtest_results[name]["path"]["drawdown"].plot(ax=axes[1], lw=1.4, label=name)
        plotted.append(name)
axes[0].set_title("Selected Layer 3 wealth curves")
axes[1].set_title("Selected Layer 3 drawdowns")
axes[1].axhline(0.0, color="black", lw=1, alpha=0.5)
axes[0].legend(loc="upper left", ncol=2)
axes[1].legend(loc="lower left", ncol=2)
plt.tight_layout()
plt.show()

print("Out-of-sample method summary")
print(comparison_df.sort_values("sharpe", ascending=False).round(4).to_string(index=False))
if not benchmark_df.empty:
    print()
    print("Benchmark summary")
    print(benchmark_df.round(4).to_string(index=False))


## 7. Sensitivity, Stability, and Why the Winner Should Not Be Chosen by Sharpe Alone

A portfolio construction method can look good for the wrong reasons:

- it is highly concentrated
- it depends on one unstable expected-return input
- it only works under one covariance window
- it earns its Sharpe by taking unpleasant tail risk
- it turns over too aggressively to be practical

The notebook therefore adds five simple stability checks:

1. **Expected-return sensitivity** for mean-sensitive allocators
2. **Covariance-window sensitivity** for selected methods
3. **Stacked dampener sensitivity** for target-vol caps, reallocation speed, and regime overlays
4. **Black-Litterman confidence calibration sanity checks**
5. **Subperiod / regime stability** rather than a single full-sample score

This is also the section where we explicitly resist the temptation to crown the most complex optimizer by default. The dampener test is deliberately small: it keeps the baseline defaults unchanged and asks whether the current overlay stack is obviously underdeploying capital.


In [ ]:
sensitivity_specs = [
    {
        "label": "mvo_equal",
        "engine": "mvo",
        "expected_return_key": "equal_signal",
        "covariance_method": "ledoit_wolf",
        "train_window_weeks": TRAIN_WINDOW_WEEKS,
    },
    {
        "label": "mvo_ic",
        "engine": "mvo",
        "expected_return_key": "ic_signal",
        "covariance_method": "ledoit_wolf",
        "train_window_weeks": TRAIN_WINDOW_WEEKS,
    },
    {
        "label": "mvo_regime",
        "engine": "mvo",
        "expected_return_key": "regime_signal",
        "covariance_method": "ledoit_wolf",
        "train_window_weeks": TRAIN_WINDOW_WEEKS,
    },
    {
        "label": "min_var_104w",
        "engine": "min_variance",
        "expected_return_key": None,
        "covariance_method": "ledoit_wolf",
        "train_window_weeks": ALT_TRAIN_WINDOW_WEEKS,
    },
    {
        "label": "hrp_104w",
        "engine": "hrp",
        "expected_return_key": None,
        "covariance_method": "sample",
        "train_window_weeks": ALT_TRAIN_WINDOW_WEEKS,
    },
]

sensitivity_rows = []
for spec in sensitivity_specs:
    sleeve_alloc, etf_weights, path, diagnostics = run_method_backtest(
        method_name=spec["label"],
        method_category="sensitivity",
        engine=spec["engine"],
        sleeve_return_panel=sleeve_return_panel,
        sleeve_positions=sleeve_positions,
        forward_weekly_returns=next_week_returns.reindex(columns=weekly_prices.columns),
        conviction_inputs=conviction_inputs,
        regime_states=regime_states,
        cash_proxy=cash_proxy,
        train_window_weeks=spec["train_window_weeks"],
        expected_return_key=spec["expected_return_key"],
        covariance_method=spec["covariance_method"],
    )
    metrics = summary_metrics(
        path["net_return"],
        turnover_series=path["turnover"],
        weight_panel=etf_weights,
        allocation_panel=sleeve_alloc,
        trials=len(method_specs),
    )
    metrics.update(spec)
    sensitivity_rows.append(metrics)

sensitivity_df = pd.DataFrame(sensitivity_rows)


def build_dampener_regime_states(regime_states, overlay_variant):
    adjusted = regime_states.copy()
    if adjusted.empty or overlay_variant == "baseline" or "overlay_multiplier" not in adjusted.columns:
        return adjusted
    if overlay_variant == "looser_neutral_stress":
        if "risk_state" in adjusted.columns:
            neutral_mask = adjusted["risk_state"].eq("neutral")
            stressed_mask = adjusted["risk_state"].eq("stressed")
            adjusted.loc[neutral_mask, "overlay_multiplier"] = adjusted.loc[neutral_mask, "overlay_multiplier"].clip(lower=0.80)
            adjusted.loc[stressed_mask, "overlay_multiplier"] = adjusted.loc[stressed_mask, "overlay_multiplier"].clip(lower=0.40)
        else:
            adjusted["overlay_multiplier"] = adjusted["overlay_multiplier"].clip(lower=0.80)
    return adjusted


dampener_specs = [
    {
        "label": f"erc_ceil_{ceil:.2f}_speed_{speed:.2f}",
        "target_vol_ceil": ceil,
        "sleeve_reallocation_speed": speed,
        "overlay_variant": "baseline",
    }
    for ceil in [1.00, 1.10, 1.25]
    for speed in [0.40, 0.60, 1.00]
]
dampener_specs.append(
    {
        "label": "erc_looser_overlay_ceil_1.00_speed_0.60",
        "target_vol_ceil": 1.00,
        "sleeve_reallocation_speed": 0.60,
        "overlay_variant": "looser_neutral_stress",
    }
)

dampener_rows = []
for spec in dampener_specs:
    test_regime_states = build_dampener_regime_states(regime_states, spec["overlay_variant"])
    sleeve_alloc, etf_weights, path, diagnostics = run_method_backtest(
        method_name=spec["label"],
        method_category="dampener_sensitivity",
        engine="erc",
        sleeve_return_panel=sleeve_return_panel,
        sleeve_positions=sleeve_positions,
        forward_weekly_returns=next_week_returns.reindex(columns=weekly_prices.columns),
        conviction_inputs=conviction_inputs,
        regime_states=test_regime_states,
        cash_proxy=cash_proxy,
        train_window_weeks=TRAIN_WINDOW_WEEKS,
        expected_return_key=None,
        covariance_method="ledoit_wolf",
        target_vol_ceil=spec["target_vol_ceil"],
        sleeve_reallocation_speed=spec["sleeve_reallocation_speed"],
    )
    metrics = summary_metrics(
        path["net_return"],
        turnover_series=path["turnover"],
        weight_panel=etf_weights,
        allocation_panel=sleeve_alloc,
        trials=len(method_specs),
    )
    for diagnostic_col in ["avg_gross_multiplier", "avg_cash_weight", "avg_target_vol_multiplier", "avg_regime_multiplier"]:
        metrics[diagnostic_col] = np.nan
    if not diagnostics.empty:
        metrics["avg_gross_multiplier"] = diagnostics["gross_multiplier"].mean()
        metrics["avg_cash_weight"] = diagnostics["cash_weight"].mean()
        metrics["avg_target_vol_multiplier"] = diagnostics["target_vol_multiplier"].mean()
        metrics["avg_regime_multiplier"] = diagnostics["regime_multiplier"].mean()
    metrics.update(spec)
    dampener_rows.append(metrics)

dampener_sensitivity_df = pd.DataFrame(dampener_rows)
if not dampener_sensitivity_df.empty:
    default_mask = (
        dampener_sensitivity_df["target_vol_ceil"].eq(TARGET_VOL_CEIL)
        & dampener_sensitivity_df["sleeve_reallocation_speed"].eq(SLEEVE_REALLOCATION_SPEED)
        & dampener_sensitivity_df["overlay_variant"].eq("baseline")
    )
    if default_mask.any():
        default_row = dampener_sensitivity_df.loc[default_mask].iloc[0]
        for col in ["ann_return", "ann_vol", "sharpe", "max_drawdown", "avg_cash_weight", "avg_gross_multiplier"]:
            if col in dampener_sensitivity_df.columns:
                dampener_sensitivity_df[f"{col}_delta_vs_default"] = dampener_sensitivity_df[col] - default_row[col]

bl_confidence_sensitivity_df = pd.DataFrame(
    [
        {
            "view_spread": spread,
            "confidence_floor": floor,
            "confidence": black_litterman_confidence_from_spread(spread, floor=floor),
            "spread_divisor": BL_CONFIDENCE_SPREAD_DIVISOR,
            "confidence_ceiling": BL_CONFIDENCE_CEIL,
        }
        for floor in [0.10, BL_CONFIDENCE_FLOOR]
        for spread in [0.00, 0.25, 0.50, 1.00, 2.00]
    ]
)

static_rows = []
for spec in method_specs:
    weights_static = static_in_sample_weights(
        engine=spec["engine"],
        sleeve_return_panel=sleeve_return_panel.dropna(how="any"),
        conviction_inputs=conviction_inputs,
        expected_return_key=spec["expected_return_key"],
        covariance_method=spec["covariance_method"],
    )
    if weights_static.empty:
        continue
    static_returns = (sleeve_return_panel[weights_static.index].fillna(0.0) * weights_static).sum(axis=1)
    static_metrics = summary_metrics(static_returns, trials=len(method_specs))
    static_metrics.update({"method_name": spec["method_name"], "category": spec["category"]})
    static_rows.append(static_metrics)

static_in_sample_df = pd.DataFrame(static_rows)

print("Sensitivity checks")
print(sensitivity_df[["label", "ann_return", "ann_vol", "sharpe", "max_drawdown", "avg_weekly_turnover"]].round(4).to_string(index=False))
print()
print("Stacked dampener sensitivity using ERC as a representative robust allocator")
print(dampener_sensitivity_df[[
    "label",
    "target_vol_ceil",
    "sleeve_reallocation_speed",
    "overlay_variant",
    "ann_return",
    "ann_vol",
    "sharpe",
    "max_drawdown",
    "avg_gross_multiplier",
    "avg_cash_weight",
]].round(4).to_string(index=False))
print()
print("Black-Litterman confidence mapping sanity check")
print(bl_confidence_sensitivity_df.round(4).to_string(index=False))
print()
print("Static in-sample optimism check")
print(static_in_sample_df.sort_values("sharpe", ascending=False)[["method_name", "ann_return", "ann_vol", "sharpe", "max_drawdown"]].round(4).to_string(index=False))


## 8. Improvement Lab — Diagnosing Upside Leakage, Testing Recovery Re-Risking, and Comparing Finalists

This section now focuses not only on reducing excessive defensiveness, but also on recovering benchmark participation faster after stress while keeping the drawdown discipline that made the earlier versions credible.


In [ ]:
selective_positions_path = layer2a_dir / "strategy_positions_composite_selective_signals.csv"
selective_returns_path = layer2a_dir / "strategy_returns_composite_selective_signals.csv"

selective_positions = read_strategy_positions(selective_positions_path)
selective_returns = read_strategy_returns(selective_returns_path)

if selective_positions.empty or selective_returns.empty:
    raise ValueError(
        "The selective composite sleeve is missing. Run 03_layer2a_strategy_logic.ipynb first so "
        "composite_selective_signals is available for the improvement lab."
    )

enhanced_sleeve_positions = dict(sleeve_positions)
enhanced_sleeve_positions["composite_selective_signals"] = selective_positions.reindex(index=weekly_prices.index).ffill().fillna(0.0)

enhanced_sleeve_return_panel = sleeve_return_panel.copy()
enhanced_sleeve_return_panel["composite_selective_signals"] = (
    selective_returns["net_return"] if "net_return" in selective_returns.columns else selective_returns.iloc[:, 0]
).reindex(weekly_prices.index).fillna(0.0)

baseline_subset = list(sleeve_return_panel.columns)
drop_breadth_subset = [name for name in baseline_subset if name != "composite_breadth_filtered"]
replace_equal_subset = [
    "dual_momentum_topn",
    "cta_trend_long_only",
    "composite_selective_signals",
    "composite_regime_conditioned",
    "taa_10m_sma",
]


def classify_allocations(weight_panel, cash_proxy):
    defensive_assets = [ticker for ticker in ["IEF", "SHY", "TLT", "TIP", "GLD"] if ticker in weight_panel.columns and ticker != cash_proxy]
    offensive_assets = [ticker for ticker in weight_panel.columns if ticker not in set(defensive_assets + [cash_proxy])]
    return offensive_assets, defensive_assets


def version_state_label(current_offensive, current_defensive, current_cash):
    if current_cash + current_defensive >= 0.55:
        return "defensive"
    if current_offensive >= 0.60 and current_cash <= 0.20:
        return "risk_on"
    return "neutral"


def run_improvement_subset(method_name, subset_name, subset_sleeves, overlay_variant="baseline", sleeve_reallocation_speed=SLEEVE_REALLOCATION_SPEED, target_vol_ceil=TARGET_VOL_CEIL):
    method_spec = next(spec for spec in method_specs if spec["method_name"] == method_name)
    subset = [name for name in subset_sleeves if name in enhanced_sleeve_return_panel.columns]
    subset_regime_states = build_dampener_regime_states(regime_states, overlay_variant)
    sleeve_alloc, etf_weights, path, diagnostics = run_method_backtest(
        method_name=f"{subset_name}_{method_name}",
        method_category="improvement_lab",
        engine=method_spec["engine"],
        sleeve_return_panel=enhanced_sleeve_return_panel[subset],
        sleeve_positions={name: enhanced_sleeve_positions[name] for name in subset},
        forward_weekly_returns=next_week_returns.reindex(columns=weekly_prices.columns),
        conviction_inputs={key: value.reindex(columns=[name for name in subset if name in value.columns]) for key, value in conviction_inputs.items()},
        regime_states=subset_regime_states,
        cash_proxy=cash_proxy,
        train_window_weeks=TRAIN_WINDOW_WEEKS,
        expected_return_key=method_spec["expected_return_key"],
        covariance_method=method_spec["covariance_method"],
        target_vol_ceil=target_vol_ceil,
        sleeve_reallocation_speed=sleeve_reallocation_speed,
    )
    metrics = summary_metrics(
        path["net_return"],
        turnover_series=path["turnover"],
        weight_panel=etf_weights,
        allocation_panel=sleeve_alloc,
        trials=max(len(subset), 2),
    )
    return sleeve_alloc, etf_weights, path, diagnostics, metrics


sleeve_subset_specs = {
    "baseline_current": baseline_subset,
    "drop_breadth": drop_breadth_subset,
    "drop_regime": [name for name in baseline_subset if name != "composite_regime_conditioned"],
    "replace_equal_with_selective": replace_equal_subset,
    "add_selective_drop_breadth": drop_breadth_subset + ["composite_selective_signals"],
}

sleeve_subset_rows = []
sleeve_incremental_rows = []
baseline_rows = {}

strategy_lookup = strategy_summary.set_index("strategy_name") if not strategy_summary.empty else pd.DataFrame()

for method_name in ["hrp", "max_diversification"]:
    _, baseline_weights, _, baseline_diag, baseline_metrics = run_improvement_subset(method_name, "baseline_current", baseline_subset)
    baseline_rows[method_name] = {
        "metrics": baseline_metrics,
        "avg_bil_weight": baseline_weights.get(cash_proxy, pd.Series(dtype=float)).mean() if cash_proxy in baseline_weights.columns else np.nan,
        "avg_cash_weight": baseline_diag["cash_weight"].mean() if not baseline_diag.empty else np.nan,
    }

    for subset_name, subset_sleeves in sleeve_subset_specs.items():
        _, weight_panel, path, diagnostics, metrics = run_improvement_subset(method_name, subset_name, subset_sleeves)
        row = {
            "method_name": method_name,
            "subset_name": subset_name,
            "sleeve_count": len(subset_sleeves),
            "sleeve_names": "|".join(subset_sleeves),
            **metrics,
            "avg_bil_weight": weight_panel.get(cash_proxy, pd.Series(dtype=float)).mean() if cash_proxy in weight_panel.columns else np.nan,
            "avg_spy_weight": weight_panel.get(market_proxy, pd.Series(dtype=float)).mean() if market_proxy in weight_panel.columns else np.nan,
            "avg_cash_weight": diagnostics["cash_weight"].mean() if not diagnostics.empty else np.nan,
        }
        baseline = baseline_rows[method_name]["metrics"]
        row["delta_ann_return_vs_baseline"] = row["ann_return"] - baseline["ann_return"]
        row["delta_sharpe_vs_baseline"] = row["sharpe"] - baseline["sharpe"]
        row["delta_max_drawdown_vs_baseline"] = row["max_drawdown"] - baseline["max_drawdown"]
        row["delta_cvar_5_vs_baseline"] = row["cvar_5"] - baseline["cvar_5"]
        row["delta_turnover_vs_baseline"] = row["avg_weekly_turnover"] - baseline["avg_weekly_turnover"]
        row["delta_avg_bil_vs_baseline"] = row["avg_bil_weight"] - baseline_rows[method_name]["avg_bil_weight"]
        row["delta_avg_cash_vs_baseline"] = row["avg_cash_weight"] - baseline_rows[method_name]["avg_cash_weight"]
        sleeve_subset_rows.append(row)

        if subset_name != "baseline_current":
            changed_sleeves = sorted(set(baseline_subset).symmetric_difference(set(subset_sleeves)))
            for sleeve_name in changed_sleeves:
                standalone = strategy_lookup.loc[sleeve_name].to_dict() if isinstance(strategy_lookup, pd.DataFrame) and sleeve_name in strategy_lookup.index else {}
                sleeve_incremental_rows.append(
                    {
                        "method_name": method_name,
                        "subset_name": subset_name,
                        "candidate_sleeve": sleeve_name,
                        "standalone_ann_return": standalone.get("ann_return"),
                        "standalone_sharpe": standalone.get("sharpe"),
                        "standalone_max_drawdown": standalone.get("max_drawdown"),
                        "standalone_avg_weekly_turnover": standalone.get("avg_weekly_turnover"),
                        **row,
                    }
                )

portfolio_version_specs = [
    {
        "version_name": "baseline_hrp_default",
        "method_name": "hrp",
        "subset_sleeves": baseline_subset,
        "overlay_variant": "baseline",
        "sleeve_reallocation_speed": 0.40,
        "target_vol_ceil": 1.00,
        "note": "Original baseline stack with the experimental breadth sleeve still active.",
    },
    {
        "version_name": "improved_hrp_selective",
        "method_name": "hrp",
        "subset_sleeves": replace_equal_subset,
        "overlay_variant": "looser_neutral_stress",
        "sleeve_reallocation_speed": 0.60,
        "target_vol_ceil": 1.00,
        "note": "Drops the redundant breadth sleeve, swaps in the selective composite sleeve, and relaxes only the overlay that was visibly binding.",
    },
    {
        "version_name": "baseline_max_div_default",
        "method_name": "max_diversification",
        "subset_sleeves": baseline_subset,
        "overlay_variant": "baseline",
        "sleeve_reallocation_speed": 0.40,
        "target_vol_ceil": 1.00,
        "note": "Original maximum-diversification baseline.",
    },
    {
        "version_name": "improved_max_div_selective",
        "method_name": "max_diversification",
        "subset_sleeves": replace_equal_subset,
        "overlay_variant": "looser_neutral_stress",
        "sleeve_reallocation_speed": 0.60,
        "target_vol_ceil": 1.00,
        "note": "Improved maximum-diversification version using the same selective sleeve set and looser overlay rules.",
    },
    {
        "version_name": "improved_inverse_vol_selective",
        "method_name": "inverse_vol",
        "subset_sleeves": replace_equal_subset,
        "overlay_variant": "looser_neutral_stress",
        "sleeve_reallocation_speed": 0.60,
        "target_vol_ceil": 1.00,
        "note": "Inverse-vol reference run on the improved finalist sleeves.",
    },
    {
        "version_name": "improved_herc_selective",
        "method_name": "herc",
        "subset_sleeves": replace_equal_subset,
        "overlay_variant": "looser_neutral_stress",
        "sleeve_reallocation_speed": 0.60,
        "target_vol_ceil": 1.00,
        "note": "HERC reference run on the improved finalist sleeves.",
    },
]

portfolio_version_rows = []
portfolio_version_regime_rows = []
portfolio_version_subperiod_rows = []
allocation_driver_rows = []
allocation_driver_breakdown_rows = []
allocation_driver_timeseries_rows = []
version_baselines = {}

for version in portfolio_version_specs:
    sleeve_alloc, weight_panel, path, diagnostics, metrics = run_improvement_subset(
        version["method_name"],
        version["version_name"],
        version["subset_sleeves"],
        overlay_variant=version["overlay_variant"],
        sleeve_reallocation_speed=version["sleeve_reallocation_speed"],
        target_vol_ceil=version["target_vol_ceil"],
    )

    weight_panel.to_csv(OUTPUT_DIR / f"portfolio_version_weights_{version['version_name']}.csv")
    sleeve_alloc.to_csv(OUTPUT_DIR / f"portfolio_version_sleeve_weights_{version['version_name']}.csv")
    path.to_csv(OUTPUT_DIR / f"portfolio_version_returns_{version['version_name']}.csv")

    regime_split = regime_split_summary(version["version_name"], path["net_return"], regime_states.get("risk_state", pd.Series(dtype=object)))
    subperiod_split = subperiod_summary(version["version_name"], path["net_return"])
    if not regime_split.empty:
        portfolio_version_regime_rows.append(regime_split)
    if not subperiod_split.empty:
        portfolio_version_subperiod_rows.append(subperiod_split)

    version_row = {
        **version,
        **metrics,
        "avg_bil_weight": weight_panel.get(cash_proxy, pd.Series(dtype=float)).mean() if cash_proxy in weight_panel.columns else np.nan,
        "avg_spy_weight": weight_panel.get(market_proxy, pd.Series(dtype=float)).mean() if market_proxy in weight_panel.columns else np.nan,
        "avg_cash_weight": diagnostics["cash_weight"].mean() if not diagnostics.empty else np.nan,
        "avg_regime_multiplier": diagnostics["regime_multiplier"].mean() if not diagnostics.empty else np.nan,
        "avg_target_vol_multiplier": diagnostics["target_vol_multiplier"].mean() if not diagnostics.empty else np.nan,
        "avg_gross_multiplier": diagnostics["gross_multiplier"].mean() if not diagnostics.empty else np.nan,
    }
    if version["version_name"].startswith("baseline_"):
        version_baselines[version["method_name"]] = version_row
    elif version["method_name"] in version_baselines:
        baseline = version_baselines[version["method_name"]]
        for key in ["ann_return", "ann_vol", "sharpe", "max_drawdown", "calmar", "cvar_5", "avg_weekly_turnover", "avg_effective_n", "avg_bil_weight", "avg_spy_weight", "avg_cash_weight"]:
            version_row[f"delta_{key}_vs_baseline"] = version_row[key] - baseline[key]
    portfolio_version_rows.append(version_row)

    offensive_assets, defensive_assets = classify_allocations(weight_panel, cash_proxy)
    offensive_weight = weight_panel.reindex(columns=offensive_assets, fill_value=0.0).sum(axis=1)
    defensive_weight = weight_panel.reindex(columns=defensive_assets, fill_value=0.0).sum(axis=1)
    cash_weight = weight_panel.get(cash_proxy, pd.Series(0.0, index=weight_panel.index))
    overlay_cash = sleeve_alloc.get(f"cash::{cash_proxy}", pd.Series(0.0, index=weight_panel.index))
    sleeve_bil = (cash_weight - overlay_cash).clip(lower=0.0)
    latest_date = weight_panel.index[-1]

    allocation_driver_rows.append(
        {
            "version_name": version["version_name"],
            "method_name": version["method_name"],
            "current_date": latest_date.date().isoformat(),
            "current_risk_state": regime_states.loc[latest_date, "risk_state"] if latest_date in regime_states.index and "risk_state" in regime_states.columns else None,
            "current_state_label": version_state_label(offensive_weight.loc[latest_date], defensive_weight.loc[latest_date], cash_weight.loc[latest_date]),
            "current_offensive_weight": offensive_weight.loc[latest_date],
            "current_defensive_weight": defensive_weight.loc[latest_date],
            "current_cash_proxy_weight": cash_weight.loc[latest_date],
            "current_bil_weight": cash_weight.loc[latest_date],
            "current_spy_weight": weight_panel.loc[latest_date].get(market_proxy, np.nan),
            "avg_offensive_weight": offensive_weight.mean(),
            "avg_defensive_weight": defensive_weight.mean(),
            "avg_cash_proxy_weight": cash_weight.mean(),
            "avg_bil_weight": cash_weight.mean(),
            "avg_spy_weight": weight_panel.get(market_proxy, pd.Series(dtype=float)).mean() if market_proxy in weight_panel.columns else np.nan,
            "avg_overlay_cash_weight": overlay_cash.mean(),
            "avg_sleeve_bil_weight": sleeve_bil.mean(),
            "current_overlay_cash_weight": overlay_cash.loc[latest_date],
            "current_sleeve_bil_weight": sleeve_bil.loc[latest_date],
            "avg_target_vol_multiplier": diagnostics["target_vol_multiplier"].mean() if not diagnostics.empty else np.nan,
            "avg_regime_multiplier": diagnostics["regime_multiplier"].mean() if not diagnostics.empty else np.nan,
            "avg_gross_multiplier": diagnostics["gross_multiplier"].mean() if not diagnostics.empty else np.nan,
            "calm_regime_frequency": regime_states.get("risk_state", pd.Series(dtype=object)).eq("calm").mean(),
            "neutral_regime_frequency": regime_states.get("risk_state", pd.Series(dtype=object)).eq("neutral").mean(),
            "stressed_regime_frequency": regime_states.get("risk_state", pd.Series(dtype=object)).eq("stressed").mean(),
        }
    )

    for date in weight_panel.index:
        allocation_driver_timeseries_rows.append(
            {
                "Date": date.date().isoformat(),
                "version_name": version["version_name"],
                "offensive_weight": offensive_weight.loc[date],
                "defensive_weight": defensive_weight.loc[date],
                "cash_proxy_weight": cash_weight.loc[date],
                "bil_weight": cash_weight.loc[date],
                "spy_weight": weight_panel.loc[date].get(market_proxy, np.nan),
                "overlay_cash_weight": overlay_cash.loc[date],
                "sleeve_bil_weight": sleeve_bil.loc[date],
                "risk_state": regime_states.loc[date, "risk_state"] if date in regime_states.index and "risk_state" in regime_states.columns else None,
            }
        )

    current_sleeve_alloc = sleeve_alloc.loc[latest_date] if latest_date in sleeve_alloc.index else pd.Series(dtype=float)
    for asset in [cash_proxy, market_proxy]:
        allocation_driver_breakdown_rows.append(
            {
                "version_name": version["version_name"],
                "horizon": "current",
                "asset": asset,
                "driver": "overlay_cash",
                "contribution": current_sleeve_alloc.get(f"cash::{cash_proxy}", 0.0) if asset == cash_proxy else 0.0,
            }
        )
        for sleeve_name in [name for name in current_sleeve_alloc.index if not str(name).startswith("cash::")]:
            sleeve_weight = current_sleeve_alloc.get(sleeve_name, 0.0)
            sleeve_position = enhanced_sleeve_positions[sleeve_name].loc[latest_date].get(asset, 0.0) if latest_date in enhanced_sleeve_positions[sleeve_name].index else 0.0
            allocation_driver_breakdown_rows.append(
                {
                    "version_name": version["version_name"],
                    "horizon": "current",
                    "asset": asset,
                    "driver": sleeve_name,
                    "contribution": sleeve_weight * sleeve_position,
                }
            )

sleeve_incremental_df = pd.DataFrame(sleeve_incremental_rows)
sleeve_subset_df = pd.DataFrame(sleeve_subset_rows)
portfolio_version_df = pd.DataFrame(portfolio_version_rows)
allocation_driver_df = pd.DataFrame(allocation_driver_rows)

sleeve_incremental_path = OUTPUT_DIR / "sleeve_incremental_contribution.csv"
sleeve_subset_path = OUTPUT_DIR / "sleeve_subset_comparison.csv"
portfolio_version_path = OUTPUT_DIR / "portfolio_version_comparison.csv"
portfolio_version_regime_path = OUTPUT_DIR / "portfolio_version_regime_split_summary.csv"
portfolio_version_subperiod_path = OUTPUT_DIR / "portfolio_version_subperiod_summary.csv"
allocation_driver_path = OUTPUT_DIR / "allocation_driver_summary.csv"
allocation_breakdown_path = OUTPUT_DIR / "allocation_driver_breakdown.csv"
allocation_timeseries_path = OUTPUT_DIR / "allocation_driver_timeseries.csv"

sleeve_incremental_df.to_csv(sleeve_incremental_path, index=False)
sleeve_subset_df.to_csv(sleeve_subset_path, index=False)
portfolio_version_df.to_csv(portfolio_version_path, index=False)
pd.concat(portfolio_version_regime_rows, ignore_index=True).to_csv(portfolio_version_regime_path, index=False)
pd.concat(portfolio_version_subperiod_rows, ignore_index=True).to_csv(portfolio_version_subperiod_path, index=False)
allocation_driver_df.to_csv(allocation_driver_path, index=False)
pd.DataFrame(allocation_driver_breakdown_rows).to_csv(allocation_breakdown_path, index=False)
pd.DataFrame(allocation_driver_timeseries_rows).to_csv(allocation_timeseries_path, index=False)

print(f"Saved: {sleeve_incremental_path}")
print(f"Saved: {sleeve_subset_path}")
print(f"Saved: {portfolio_version_path}")
print(f"Saved: {allocation_driver_path}")
print()
print("Improvement lab version comparison")
print(portfolio_version_df.round(4).to_string(index=False))
print()
print("Current allocation driver summary")
print(allocation_driver_df.round(4).to_string(index=False))


## 8. Final Comparison Framework

The notebook does **not** choose a winner from the highest Sharpe alone.

Instead, the final ranking blends:

- out-of-sample Sharpe
- drawdown control
- tail-risk control
- turnover
- concentration / diversification
- weight stability
- subperiod robustness

This is closer to how a production allocator should be chosen. A method that gives up a little headline Sharpe in exchange for better drawdown behavior, lower turnover, and fewer concentration spikes is often the better default.


In [ ]:
comparison_methods = comparison_df.copy()
if not comparison_methods.empty:
    subperiod_range = subperiod_df.groupby("method_name")["sharpe"].agg(lambda s: s.max() - s.min() if len(s.dropna()) else np.nan)
    comparison_methods["subperiod_sharpe_range"] = comparison_methods["method_name"].map(subperiod_range)
    comparison_methods["instability_flag"] = (
        comparison_methods["avg_max_weight"].fillna(0.0).gt(0.45)
        | comparison_methods["avg_weekly_turnover"].fillna(0.0).gt(0.35)
        | comparison_methods["weight_instability"].fillna(0.0).gt(0.80)
    )

    def pct_rank(series, ascending=True):
        series = pd.Series(series)
        return series.rank(pct=True, ascending=ascending)

    comparison_methods["rank_sharpe"] = pct_rank(comparison_methods["sharpe"], ascending=True)
    comparison_methods["rank_drawdown"] = pct_rank(comparison_methods["max_drawdown"], ascending=True)
    comparison_methods["rank_cvar"] = pct_rank(comparison_methods["cvar_5"], ascending=True)
    comparison_methods["rank_turnover"] = pct_rank(-comparison_methods["avg_weekly_turnover"], ascending=True)
    comparison_methods["rank_diversification"] = pct_rank(comparison_methods["avg_effective_n"], ascending=True)
    comparison_methods["rank_stability"] = pct_rank(-comparison_methods["subperiod_sharpe_range"], ascending=True)
    comparison_methods["rank_concentration"] = pct_rank(-comparison_methods["avg_max_weight"], ascending=True)

    comparison_methods["robustness_score"] = (
        0.25 * comparison_methods["rank_sharpe"].fillna(0.0)
        + 0.20 * comparison_methods["rank_drawdown"].fillna(0.0)
        + 0.15 * comparison_methods["rank_cvar"].fillna(0.0)
        + 0.15 * comparison_methods["rank_turnover"].fillna(0.0)
        + 0.10 * comparison_methods["rank_diversification"].fillna(0.0)
        + 0.10 * comparison_methods["rank_stability"].fillna(0.0)
        + 0.05 * comparison_methods["rank_concentration"].fillna(0.0)
    )
    comparison_methods = comparison_methods.sort_values(["robustness_score", "sharpe"], ascending=[False, False]).reset_index(drop=True)

best_oos = comparison_methods.sort_values("sharpe", ascending=False).head(1)
best_drawdown = comparison_methods.sort_values("max_drawdown", ascending=False).head(1)
best_low_turnover = comparison_methods.sort_values(["avg_weekly_turnover", "robustness_score"], ascending=[True, False]).head(1)
best_default = comparison_methods[~comparison_methods["instability_flag"]].sort_values(["robustness_score", "avg_weekly_turnover"], ascending=[False, True]).head(1)
best_in_sample = static_in_sample_df.sort_values("sharpe", ascending=False).head(1)

fig, ax = plt.subplots(figsize=(12, 6))
if not comparison_methods.empty:
    bars = comparison_methods.set_index("method_name")["robustness_score"].sort_values(ascending=True)
    bars.plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title("Layer 3 robustness ranking")
    ax.set_xlabel("Composite robustness score")
plt.tight_layout()
plt.show()

print("Best in-sample method")
print(best_in_sample[["method_name", "sharpe", "max_drawdown"]].round(4).to_string(index=False))
print()
print("Best out-of-sample method by Sharpe")
print(best_oos[["method_name", "sharpe", "max_drawdown", "avg_weekly_turnover"]].round(4).to_string(index=False))
print()
print("Best drawdown-controlled method")
print(best_drawdown[["method_name", "sharpe", "max_drawdown", "cvar_5"]].round(4).to_string(index=False))
print()
print("Best low-turnover practical method")
print(best_low_turnover[["method_name", "sharpe", "avg_weekly_turnover", "avg_max_weight"]].round(4).to_string(index=False))
print()
print("Best default production candidate")
print(best_default[["method_name", "robustness_score", "sharpe", "max_drawdown", "avg_weekly_turnover"]].round(4).to_string(index=False))
print()
print("Full method comparison")
print(comparison_methods.round(4).to_string(index=False))


## 9. Save Outputs for Downstream Use

The goal is not just to print a comparison table once. Layer 3 should leave behind clean artifacts that later notebooks or scripts can consume.

The notebook therefore saves:

- ETF weight histories by method
- sleeve weight histories by method
- return paths by method
- summary comparison tables
- cost, dampener, Black-Litterman confidence, and stability diagnostics
- a machine-readable Layer 3 manifest


In [ ]:
for method_name, payload in backtest_results.items():
    payload["weights"].to_csv(OUTPUT_DIR / f"portfolio_weights_{method_name}.csv")
    payload["sleeve_alloc"].to_csv(OUTPUT_DIR / f"portfolio_sleeve_weights_{method_name}.csv")
    payload["path"].to_csv(OUTPUT_DIR / f"portfolio_returns_{method_name}.csv")

metrics_summary_path = OUTPUT_DIR / "portfolio_metrics_summary.csv"
comparison_path = OUTPUT_DIR / "portfolio_method_comparison.csv"
cost_path = OUTPUT_DIR / "portfolio_cost_sensitivity.csv"
regime_path = OUTPUT_DIR / "portfolio_regime_split_summary.csv"
subperiod_path = OUTPUT_DIR / "portfolio_subperiod_summary.csv"
sensitivity_path = OUTPUT_DIR / "portfolio_sensitivity_checks.csv"
dampener_sensitivity_path = OUTPUT_DIR / "portfolio_dampener_sensitivity.csv"
bl_confidence_path = OUTPUT_DIR / "portfolio_bl_confidence_sensitivity.csv"
static_path = OUTPUT_DIR / "portfolio_in_sample_static_summary.csv"
conviction_path = OUTPUT_DIR / "expected_return_inputs.csv"
sleeve_universe_path = OUTPUT_DIR / "portfolio_candidate_sleeves.csv"
diagnostics_path = OUTPUT_DIR / "portfolio_diagnostics.csv"
manifest_path = OUTPUT_DIR / "layer3_manifest.json"

comparison_df.to_csv(metrics_summary_path, index=False)
comparison_methods.to_csv(comparison_path, index=False)
cost_df.to_csv(cost_path, index=False)
regime_df.to_csv(regime_path, index=False)
subperiod_df.to_csv(subperiod_path, index=False)
sensitivity_df.to_csv(sensitivity_path, index=False)
dampener_sensitivity_df.to_csv(dampener_sensitivity_path, index=False)
bl_confidence_sensitivity_df.to_csv(bl_confidence_path, index=False)
static_in_sample_df.to_csv(static_path, index=False)
conviction_long.to_csv(conviction_path, index=False)
sleeve_universe.to_csv(sleeve_universe_path, index=False)
diagnostic_df.to_csv(diagnostics_path, index=False)
manifest_path.write_text(json.dumps(manifest_records, indent=2))

print(f"Saved: {metrics_summary_path}")
print(f"Saved: {comparison_path}")
print(f"Saved: {cost_path}")
print(f"Saved: {regime_path}")
print(f"Saved: {subperiod_path}")
print(f"Saved: {sensitivity_path}")
print(f"Saved: {dampener_sensitivity_path}")
print(f"Saved: {bl_confidence_path}")
print(f"Saved: {static_path}")
print(f"Saved: {conviction_path}")
print(f"Saved: {sleeve_universe_path}")
print(f"Saved: {diagnostics_path}")
print(f"Saved: {manifest_path}")
print()
print("Files now in the Layer 3 output directory:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print(" -", path.name)


## 10. Final Summary

1. **Which portfolio construction methods were implemented**
   Equal weight, inverse vol, minimum variance, constrained mean-variance, constrained maximum Sharpe, a practical Black-Litterman sleeve allocator, equal risk contribution, HRP, HERC, maximum diversification, and CVaR minimization when `cvxpy` is available.

2. **Which methods look strongest out of sample**
   The notebook is designed so the saved comparison table answers this directly, but the methods that usually deserve the closest attention are the simpler or shrinkage-heavy ones: inverse vol, minimum variance, ERC, HRP, HERC, and any Black-Litterman variant that remains well-diversified after constraints.

3. **Which methods are most stable and practical**
   In ETF sleeve settings, the most practical candidates are usually the slower, constrained allocators with strong regularization and modest turnover: equal weight, inverse vol, ERC, HRP, HERC, and sometimes maximum diversification.

4. **Which methods are too sensitive to inputs**
   Classical mean-variance and tangency-style portfolios remain the most input-sensitive, especially when expected returns move around more than the sleeves themselves. Their job in this notebook is to be compared honestly against robust baselines, not assumed to win.

5. **Which methods are the best candidates for the project's default production allocator**
   The default production candidate should usually come from the high-robustness, low-instability corner of the comparison table rather than from the highest standalone Sharpe. In many research stacks like this one, that means a hierarchical or risk-budgeting method with explicit overlays often earns the default role before a more aggressive optimizer does.

6. **What evidence supports that conclusion**
   The decision should be based on the joint evidence in `portfolio_method_comparison.csv`: out-of-sample Sharpe, drawdown, CVaR, turnover, effective diversification, concentration, subperiod stability, and regime-split behavior. The notebook is intentionally set up so no single metric gets to choose the winner alone.
